# LAB 01 · Прикладная статистика

**ИТМО · Совместное исследование Инфохимии и КТвД · 2026**

### От отклика газовых сенсоров — к проверяемым статистическим выводам

**Инфохимики:** Молеваник Алексей (502512), Фатхуллина София (502766), Лемешкина Полина (504704).  
**КТвД:** Макарова Юлия (501809), Степанова Дарья (504599).

**Цель.** Исследовать распределения трёх характеристик сенсорных откликов, проверить зависимости и качество выборочных оценок, реализовать генераторы случайных значений и представить результаты в единой интерактивной системе.

**Основной источник:** [Gas Sensor Array Drift at Different Concentrations, UCI](https://archive.ics.uci.edu/dataset/270/gas+sensor+array+drift+at+different+concentrations) (13 910 измерений, 10 партий, 128 признаков). В соответствии с официальной кодировкой UCI для анализа выбран **класс 5 — ацетон** (3 009 измерений); воспроизводимая выборка: 2 500 наблюдений, `seed=42`, без возвращения.

**Дополнительное исследование КТвД:** [Diamonds / ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html) — масса и цена бриллиантов, EDA, доказательство зависимости, генерация пар.

**Структура:** [Спринт 1](#scrollTo=dc0e3bfa) · [Интерактивный атлас](#scrollTo=3b116392) · [Спринт 2](#scrollTo=1bcac0f6) · [Diamonds](#scrollTo=diamonds-intro) · [Критерии](#scrollTo=0e7069b9)

---


## Об исследовании

Газовые датчики реагируют на условия эксперимента, однако отдельное значение отклика неизвестно до выбора наблюдения. В рамках эмпирической модели случайный эксперимент — равновероятный выбор строки из доступных измерений одного газа.

**Случайные величины:** X = `feature_1`, Y = `feature_13`, Z = `feature_24`. Для всех трёх предусмотрены EDA, оценка характеристик, исследование связей, стандартизация и моделирование выборок. Параметры расчётов воспроизводимы.

**Идентификация вещества.** Официальное описание набора UCI прямо указывает: `1 = Ethanol`, `2 = Ethylene`, `3 = Ammonia`, `4 = Acetaldehyde`, `5 = Acetone`, `6 = Toluene`. Поэтому во всей данной версии измерения класса 5 обозначены как **ацетон**. В первичных текстовых заметках группы встречалась другая реконструкция кодов по числу записей в партиях; она противоречит явному ключу UCI и не используется для химической идентификации.

**Методологическое ограничение.** Выборка без возвращения позволяет изучать свойства имеющейся конечной совокупности; она не делает автоматически репрезентативными данные для всех будущих экспериментов и не гарантирует независимость строк разных партий.

[К первоисточнику UCI](https://archive.ics.uci.edu/dataset/270/gas+sensor+array+drift+at+different+concentrations)


### Запуск в Google Colab

1. Откройте этот `.ipynb` в [Google Colab](https://colab.research.google.com/) и сохраните копию на Google Диске.
2. Выберите **Среда выполнения → Выполнить все**. Блок настройки автоматически получит оригинальный архив UCI и проверит контрольные суммы десяти партий. ZIP вручную загружать не нужно.
3. CSV `diamonds` также загружается автоматически из указанного в исследовании открытого источника или резервного зеркала.
4. Раскройте или сверните код ячейки кнопкой **Показать код**. Интерактивные Plotly-графики позволяют наводить курсор, менять диапазоны и переключать серии.
5. Повторный запуск в одной сессии использует локальный кэш. После сброса среды данные будут загружены заново.

**Необходим интернет:** оба набора данных получаются из опубликованных источников. Если источник временно недоступен, Colab сообщит об этом: не подменяйте реальные наблюдения синтетическими.


### Принципы оформления

| Элемент | Решение | Зачем |
|---|---|---|
| Цветовая система | Светлый фон, холодный голубой, бирюзовый и приглушённый терракотовый | Различать данные, сравнения и статистические ориентиры |
| Гистограммы | Металлические колонны, стилизованные под трубы | Связать визуализацию с тематикой сенсоров, сохранив истинные частоты |
| Остальные графики | Лаконичные оси, тонкая сетка, контрастные линии | Не перегружать результаты декоративными элементами |
| Подписи | Название величины, условные единицы отклика, размер выборки | Сделать каждый график понятным вне контекста |
| Код и формулы | Сворачиваемые Python-ячейки; LaTeX в Markdown | Сохранить проверяемость без визуального шума |

Промышленная стилизация влияет только на внешний вид столбцов, но не на значения, диапазоны и порядок измерений. За интерактивность отвечают **Plotly** и реактивные элементы приложения **marimo**.


### Содержание · переходы

**Оригинальное исследование:** [Данные](#scrollTo=eed1b449) · [X](#scrollTo=4edb5f0b) · [Y](#scrollTo=b6f3df63) · [Z](#scrollTo=5f3e90a9) · [Математическая база](#scrollTo=abb921e8) · [Дополнительная обработка](#scrollTo=01d1e10d) · [Генераторы](#scrollTo=1bcac0f6)

**Интерактивные визуализации КТвД:** [01 Распределения](#scrollTo=dcec2215) · [02 Интервалы](#scrollTo=61b8c1e3) · [03 Партии](#scrollTo=34bb67c0) · [04 Зависимости](#scrollTo=2d37d337) · [05 Выборки](#scrollTo=7b4fe614) · [06 Модели](#scrollTo=a16dd2e7)

**Проверка сдачи:** [Автотесты](#scrollTo=5cd0d137) · [Источники](#scrollTo=c63e0600) · [Критерии](#scrollTo=0e7069b9) · [Дизайн-критерии](#scrollTo=e2fcb89b)
**Дополнительное исследование КТвД:** [Diamonds · масса, цена, EDA и генераторы](#scrollTo=diamonds-intro) · [Диаграммы Diamonds](#scrollTo=dm-section-04) · [Источник Diamonds](https://ggplot2.tidyverse.org/reference/diamonds.html)


## Содержание и навигация

- [Спринт 1 — исходное исследование без сокращений](#scrollTo=dc0e3bfa)
- [Данные и методы отбора](#scrollTo=eed1b449)
- [Случайная величина X](#scrollTo=4edb5f0b)
- [Случайная величина Y](#scrollTo=b6f3df63)
- [Случайная величина Z](#scrollTo=5f3e90a9)
- [Мат. основа, модели и зависимости](#scrollTo=abb921e8)
- [Расширенная оценка качества моделей](#scrollTo=365b68a7)
- [Дополнительная очистка и выбросы](#scrollTo=01d1e10d)
- [Гипотезы и стратификация](#scrollTo=c3b732ed)
- [Спринт 2 — генераторы](#scrollTo=1bcac0f6)
- [Сравнение генераторов](#scrollTo=1bcac0f6)
- [Автотесты](#scrollTo=5cd0d137)
- [marimo и сайт](#scrollTo=01368f1b)
- [Источники и ограничения](#scrollTo=c63e0600)
- [Критерии и ссылки на подтверждения](#scrollTo=0e7069b9)
- [Самостоятельное исследование КТвД — Diamonds](#scrollTo=diamonds-intro)
- [Интерактивные графики Diamonds](#scrollTo=dm-section-04)
- [Генератор Diamonds](#scrollTo=dm-section-18)


### Как читать интерактивные графики

- **Наведение:** точное значение наблюдения или характеристики графика.
- **Выделение области:** увеличение фрагмента; двойной щелчок возвращает исходный масштаб.
- **Легенда:** включение и скрытие отдельных серий.
- **Переключатели и ползунки:** сравнение величин, партий, объёмов выборок и моделей.

Гистограммы X, Y, Z, диагностические графики, анализ концентрации, эксперименты с выборками и модели построены из данных выбранного класса. Синтетические значения появляются только в разделе генераторов и подписаны отдельно.


# Спринт 1 · Анализ экспериментальных данных

Далее приведено исследование трёх случайных величин, включая анализ условий измерения, распределений, связей и устойчивости оценок. Технические детали расчётов доступны в свёрнутых ячейках.


### Датасет и структура эксперимента

В исследовании используются три характеристики сенсорных откликов X, Y и Z. Исходные десять партий загружаются **автоматически с UCI** при запуске Notebook. Расчёты, таблицы и интерактивные графики создаются на одних и тех же измерениях; способ формирования выборки указан в соответствующем разделе.

## Исходные измерения и подготовка таблицы

Для работы был выбран датасет [Gas Sensor Array Drift at Different Concentrations](https://archive.ics.uci.edu/dataset/270/gas+sensor+array+drift+at+different+concentrations). опубликованный в 2013 году.

Датасет состоит из десяти файлов `.dat` и суммарно содержит 13 910 экспериментальных измерений, полученных с помощью системы из 16 сенсоров.

В ходе эксперимента на сенсоры воздействовали 6 газов с разными концентрациями: ацетон, этанол, аммиак, этилен, толуол и ацетальдегид. Для каждого измерения регистрировался электрический отклик сенсора.

Каждый сенсор характеризуется 8 числовыми признаками, то есть всего $8\cdot16=128$ признаков


Импорт библиотек

In [ ]:
# @title Автоматическая настройка лабораторной (ничего загружать не нужно) { display-mode: "form" }
# Все служебные модули и автотесты встроены в этот Notebook.
# Исходные измерения скачиваются автоматически с официального UCI (dataset ID 270).
from pathlib import Path
from zipfile import ZipFile
import base64
import hashlib
import importlib.util
import io
import shutil
import subprocess
import sys
import urllib.request

IN_COLAB = importlib.util.find_spec('google.colab') is not None
ROOT = (Path('/content') if IN_COLAB else Path.cwd()) / 'LAB01_AUTORUN_2026'
ROOT.mkdir(parents=True, exist_ok=True)

# Встроенные в Notebook исходные модули проекта и тесты.
_EMBEDDED_PROJECT_ZIP_BASE64 = """..."""
with ZipFile(io.BytesIO(base64.b64decode(_EMBEDDED_PROJECT_ZIP_BASE64))) as source_files:
    source_files.extractall(ROOT)

DATA_ZIP = ROOT / 'data/raw/gas_sensor_drift.zip'
DATA_ZIP.parent.mkdir(parents=True, exist_ok=True)

# Контрольная сумма каждого исходного batch*.dat. Это гарантирует использование
# тех же измерений даже если внешний ZIP будет перепакован UCI.
EXPECTED_BATCH_SHA256 = {'batch1.dat': '6b62f3e71b90f95dd9c2157fa91ffce32571faa7f474dd52f29a4ae769144625', 'batch10.dat': 'b8309e8ff205b9d0350761989809f31f8feaefb8582aecb05f1d0f694b91184e', 'batch2.dat': '4ea9490068457badf9efd0c22f359fbfce12c208cdd440526b97ac91d31b9002', 'batch3.dat': 'aabf06bf8b8200740156235dc59458e06d99aa8faba4d6764a22c6b6c2674ff4', 'batch4.dat': 'ffc87c5146001bbe8ce90c4b2f7dad5b390ca3079f51b559032df53ffe5ff0af', 'batch5.dat': 'dcad4939c300a6c7d9f41b9470a9253eecc0ebc5c2c4e940d8ae20b8fc4240d0', 'batch6.dat': '297d3a8b594dee4e2f4d7d2c4f5c485939fa708c10b5a11ed5cce95e05049afb', 'batch7.dat': 'eb5801d38a59beaae4a2412195d8ded790175ed1611159e083649ee9fe03d150', 'batch8.dat': '58146b3e2dd68b125b2d5432cdd4968ec1fff47464ceb04282382181932d49be', 'batch9.dat': '51f368ae8dee5d285be7f2029770e10eb12f342578830bc40ea852db19d19c9b'}
DATA_SOURCES = (
    'https://archive.ics.uci.edu/static/public/270/gas%2Bsensor%2Barray%2Bdrift%2Bdataset%2Bat%2Bdifferent%2Bconcentrations.zip',
    'https://archive.ics.uci.edu/static/public/270/gas+sensor+array+drift+dataset+at+different+concentrations.zip',
)

def _data_is_correct(path):
    if not path.is_file():
        return False
    try:
        with ZipFile(path) as zip_file:
            names = set(zip_file.namelist())
            if names != set(EXPECTED_BATCH_SHA256):
                return False
            return all(hashlib.sha256(zip_file.read(name)).hexdigest() == expected
                       for name, expected in EXPECTED_BATCH_SHA256.items())
    except Exception:
        return False

if not _data_is_correct(DATA_ZIP):
    # При первом запуске Colab скачает ~9,7 МБ сам. Повторно внутри сеанса
    # файл не скачивается. Нужен доступ к интернету.
    last_errors = []
    for url in DATA_SOURCES:
        try:
            print('Получаю исходные данные UCI…')
            request = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0 (research lab; educational)'})
            tmp_file = DATA_ZIP.with_suffix('.download')
            with urllib.request.urlopen(request, timeout=180) as response, open(tmp_file,'wb') as target:
                shutil.copyfileobj(response,target)
            if not _data_is_correct(tmp_file):
                raise ValueError('Архив скачан, но измерения не совпали с исходным набором, используемым в исследовании')
            tmp_file.replace(DATA_ZIP)
            print('10 партий получены и проверены по SHA-256.')
            break
        except Exception as exc:
            last_errors.append(f'{type(exc).__name__}: {exc}')
            tmp_file.unlink(missing_ok=True)
    else:
        raise RuntimeError(
            'Не удалось автоматически скачать проверенный датасет UCI. '
            'Проверьте доступ Colab к интернету и повторите запуск этой ячейки. '
            'Подробности: ' + ' | '.join(last_errors)
        )
else:
    print('Используются ранее скачанные и проверенные исходные измерения.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Установка только недостающих зависимостей.
packages = {
    'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
    'seaborn':'seaborn', 'scipy':'scipy', 'pytest':'pytest',
    'requests':'requests', 'plotly':'plotly'
}
missing = [p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    print('Устанавливаю библиотеки:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import warnings
from scipy import stats
from src.lab_stats import (read_gas_zip, finite_population, simple_random_sample,
    quality_table, numeric_summary, z_standardize, iqr_outlier_mask,
    empirical_quantile_generator, compare_real_generated)
from src.plot_style import apply_plot_style
from src.extended_analysis import (fit_diagnostics, outlier_impact, residual_corr,
    bootstrap_mean_ci, skewnormal_generator)
apply_plot_style()
print('Готово: данные, графики, тесты и модули доступны.')
print('Папка текущей сессии:', ROOT)




---


Датасет представлен в виде 10 .dat файлов, объединяем все файлы в один DataFrame.


---



### Загрузка десяти партий

Исходные файлы `batch1.dat` — `batch10.dat` объединяются в таблицу со столбцами `batch`, `gas_id`, `concentration`, `feature_1`…`feature_128`. Архив скачивается программно; вручную загружать или распаковывать данные не требуется.

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
DATA_ZIP = ROOT / 'data/raw/gas_sensor_drift.zip'
df = read_gas_zip(DATA_ZIP)
assert df.shape == (13910, 131), f'Неожиданная форма данных: {df.shape}'
display(df.head(5))

Теперь у нас есть Датасет, который содержит все 10 .dat файлов в одном. Далее необходимо проанализировать весь массив данных и сформировать выборку данных, с которой будем работать.

Рассмотрим как распределены измерения по каждому из газов, исходя из этого сформируем выборку(возможно состоящую из измерений для одного газа)

In [ ]:
# @title Газовые классы · сверка с официальным ключом UCI { display-mode: "form" }
# Коды взяты из раздела Additional Variable Information официального UCI ID 270.
gas_names = {
    1: "Этанол", 2: "Этилен", 3: "Аммиак",
    4: "Ацетальдегид", 5: "Ацетон", 6: "Толуол"
}
df["gas_id"] = df["gas_id"].astype(int)
gas_counts = df["gas_id"].value_counts().rename(index=gas_names).sort_values(ascending=False)
from src.ktvd_embedded import gas_catalog, show
show(gas_catalog(df))


Но исходя из того, что написано в официальной документации Датасета [Gas Sensor Array Drift at Different Concentrations](https://archive.ics.uci.edu/dataset/270/gas+sensor+array+drift+at+different+concentrations), получившиеся значения не соответствуют тому, что есть в описании.


---


Проверим корректность проставления индексов. В описании указано следующее:


```
#  The first character (1-6) codes the analyte, followed by the concentration level:

1: Ethanol; 2: Ethylene; 3: Ammonia; 4: Acetaldehyde; 5: Acetone; 6: Toluene
```

---

Также указано количество измерений в каждом отдельном .dat файле:


```
Batch ID: Ethanol, Ethylene, Ammonia, Acetaldehyde, Acetone, Toluene

Batch 1: 83, 30, 70, 98, 90, 74

...

Batch 10: 600, 600, 600, 600, 600, 600
```
Сверим проставку индексов исходя из этого:




In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
print("Количество измерений в batch1:")
display(
    df[df["batch"] == 1]["gas_id"]
    .value_counts()
    .sort_index()
    .to_frame()
    .T
)

### Выбор класса и восстановление маркировки

В документации UCI приведён явный ключ `gas_id`: **1 — этанол, 2 — этилен, 3 — аммиак, 4 — ацетальдегид, 5 — ацетон, 6 — толуол**. Количества записей в опубликованной таблице партий и фактическом CSV/архиве могут восприниматься неоднозначно, поэтому переименовывать классы только по совпадению частот нельзя.

Для дальнейшего анализа выбран **класс 5 (ацетон)**: все доступные 3 009 наблюдений образуют конечную совокупность. Из неё случайно выбираются 2 500 записей **без возвращения**, `seed=42`.

Это определение класса согласовано с буквальной кодировкой исходного набора; само выделение наблюдений кодом не менялось.

[Источник: описание кода анализируемого газа, UCI](https://archive.ics.uci.edu/dataset/270/gas+sensor+array+drift+at+different+concentrations)


### Исследуемая совокупность и выборка

В работе используются три таблицы: `general` — все 13 910 строк, `population` — все 3 009 наблюдений класса 5, `sample` — простая случайная выборка объёмом 2 500 без возвращения. Для воспроизводимости задано начальное состояние генератора `seed=42`; `numeric_data` содержит числовые признаки выборки.


In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
general = df.copy()
population = finite_population(general, code=5)
sample = simple_random_sample(population, n=2500, seed=42, replace=False)
numeric_data = sample.select_dtypes(include='number')
print('Всего данных:', len(general), 'Класс 5:', len(population), 'Выборка:', len(sample))
print('Признаки и служебные столбцы:', sample.shape[1])
assert len(population) == 3009 and len(sample) == 2500
assert not sample.index.duplicated().any()

In [ ]:
# @title Подготовка ацетона · контроль качества { display-mode: "form" }
# Все 3009 наблюдений класса 5; исходные значения откликов не исправляем вручную.
acetone = general.loc[general["gas_id"].eq(5)].copy()
for column in ("concentration", "feature_1"):
    acetone[column] = pd.to_numeric(acetone[column], errors="coerce")
acetone = acetone.dropna(subset=["concentration", "feature_1"]).copy()
print("Измерений класса 5 (ацетон):", len(acetone))
print("Уникальных концентраций:", acetone["concentration"].nunique())
display(acetone.head())
quality_report = pd.DataFrame({
    "Показатель": ["Строк в выборке", "Столбцов", "Пропусков", "Бесконечных значений", "Полных дубликатов"],
    "Значение": [len(sample), sample.shape[1], int(sample.isna().sum().sum()),
                  int(np.isinf(numeric_data.to_numpy()).sum()), int(sample.duplicated().sum())],
})
display(quality_report)


In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
display(quality_table(general)); display(quality_table(sample))

**Маркировка газа.** В официальном описании UCI ID 270 код `5` соответствует **ацетону**. В этом Notebook сохраняется выделенная командой группа из 3 009 наблюдений и выборка из 2 500; отличаются только исправленные химические подписи, не значения измерений. [Официальный ключ](https://archive.ics.uci.edu/dataset/270/gas+sensor+array+drift+at+different+concentrations).

Сформировали выборку из 2 500 значений, проверили целостность данных, проверили датасет на дубликаты, пропущенные значения, перемешали данные


**Выбор трёх случайных величин**

Как было сказано в описании датасета, в исходном датасете описаны электронные отклики, полученные с каждого сенсора (по 8 откликов с 16 сенсоров).

Для дальнейшего анализа выберем характеристики, которые не являются одной и той же характеристикой разных сенсоров (так как в таком случае возможен вариант зависимости этих величин).

Поэтому для анализа выберем:


*   X - feature_1 - Максимальное изменение отклика 1го сенсора
*   Y - feature_13 - Характеристика нарастания отклика для 2го сенсора
*   Z - feature_24 - Характеристика спада отклика для 3го сенсора.

Таким образом, величины описывают разные характеристики (дельта, нарастание и спад.), поэтому есть надежда получить более интересные данные и разные характеристики для каждой из характеристик.



In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
sensor_data = sample[
    ["feature_1", "feature_13", "feature_24"]
].copy()

sensor_data.columns = ["X", "Y", "Z"]

print("Размер исследуемой таблицы:", sensor_data.shape)
display(sensor_data.head(10))

## 01 / Величина X — амплитуда отклика

Статистики, распределение, концентрация, партии и модели.


In [ ]:
# @title 01 · Фактические данные и ключевые показатели { display-mode: "form" }
from IPython.display import HTML, display
from src.lab_stats import numeric_summary
metrics = {name: numeric_summary(sample[col]) for name,col in {'X':'feature_1','Y':'feature_13','Z':'feature_24'}.items()}
# Генерация карточек из ВЫЧИСЛЕННЫХ значений.
print('Итоговая совокупность:', len(general), 'наблюдений; класс 5:',len(population), 'наблюдений')
display(HTML(f"""
<div style='display:flex;gap:12px;flex-wrap:wrap;font-family:Arial,sans-serif;margin:20px 0'>
  <div style='flex:1;min-width:155px;background:#142E3C;padding:20px;color:white;border-radius:12px'><div style='font-size:11px;letter-spacing:1.8px;color:#B6CDD9'>ВСЕ ИЗМЕРЕНИЯ</div><div style='font-size:30px;font-weight:700'>{len(general):,}</div></div>
  <div style='flex:1;min-width:155px;background:#E8F0F4;padding:20px;color:#1B3949;border-radius:12px'><div style='font-size:11px;letter-spacing:1.8px'>КЛАСС 5</div><div style='font-size:30px;font-weight:700'>{len(population):,}</div></div>
  <div style='flex:1;min-width:155px;background:#E8F0F4;padding:20px;color:#1B3949;border-radius:12px'><div style='font-size:11px;letter-spacing:1.8px'>ВЫБОРКА</div><div style='font-size:30px;font-weight:700'>{len(sample):,}</div></div>
  <div style='flex:1;min-width:155px;background:#E8F0F4;padding:20px;color:#1B3949;border-radius:12px'><div style='font-size:11px;letter-spacing:1.8px'>ПРИЗНАКИ</div><div style='font-size:30px;font-weight:700'>X / Y / Z</div></div>
</div>"""))

## Random Variable 01 — Maximum Sensor Response

### Description

В качестве первой случайной величины было выбрано максимальное изменение отклика первого сенсора - **feature_1**.



---

**Описание величины**

При воздействии газа на сенсоры электрическое сопротивление газочувствительной части датчика изменяется в ходе физико-химических процессов на его поверхности. Величина зависит от природы газа, его концентрации и других химических параметров.

Признак **feature_1** характеризует амплитуду этого отклика(сопротивления)


---

**Стохастическая природа первой величины**

В рамках исследования случайный экперимент = равновероятный выбор одного наблюдения из экспериментальной выборки.

В нашем случае - максимальное сопротивления отклика сенсора. До первого наблюдения значение заранее неизвестно, поскольку в выборке представлены данные при различных условиях (с разными веществами, концентрациями, ..., исправностью датчиков)

Следовательно, X - случайная величина относительно вероятностного эксперимента. Однако стоит сказать, что физико-химически, возможно эту величину можно описать какой-то зависимостью.


---

Практическое обоснование этого факта:

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
X = sample["feature_1"].copy()

print("Количество наблюдений:", X.count())
print("Количество уникальных значений:", X.nunique())
print("Минимальное значение:", X.min())
print("Максимальное значение:", X.max())
print("Выборочная дисперсия:", X.var())

### EDA & Distribution

Рассчитаем основные числовые характеристики и рассмотрим форму распределения.

Для исследования случайной величины X определим статистические характеристики, среднее, медиану,  дисперсию, стандартное отклонение, квартили, асимметрию и эксцесс.

Эти значения помогут определить положение и разброс значений, выявить особенности формы эмпирического распределения.

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
stats_X = pd.Series({
    "Количество наблюдений": X.count(),
    "Среднее": X.mean(),
    "Медиана": X.median(),
    "Дисперсия": X.var(ddof=1),
    "Стандартное отклонение": X.std(ddof=1),
    "Минимум": X.min(),
    "Q1 (25%)": X.quantile(0.25),
    "Q3 (75%)": X.quantile(0.75),
    "IQR": X.quantile(0.75) - X.quantile(0.25),
    "Максимум": X.max(),
    "Асимметрия": X.skew(),
    "Эксцесс": X.kurt()
})

display(
    stats_X.to_frame(name="Значение")
    .style.format("{:,.2f}", thousands=" ", decimal=",")
)

По выбранным 2 500 измерениям среднее X выше медианы, а стандартное отклонение велико по сравнению с типичным значением. Это говорит о неоднородности откликов: часть наблюдений расположена значительно выше основной массы.

Численные значения — **среднее, медиана, SD, квартили и асимметрия** — приведены в рассчитанной выше таблице. Для этой выборки асимметрия положительная, а диапазон между квартилями достаточно широк. На форму распределения могут влиять как концентрация, так и партия измерения; проверить это помогают следующие диаграммы.

Дисперсия измеряется в **квадрате единиц отклика**, стандартное отклонение — в тех же единицах, что X.


In [ ]:
# @title X · EDA: распределение, boxplot, ECDF, Q–Q { display-mode: "form" }
from IPython.display import HTML, display
from src.ktvd_embedded import metric_cards
display(HTML(metric_cards(X,"X")))
from src.ktvd_embedded import eda_panel, show
show(eda_panel(X,"X"))

**Интерактивная гистограмма X.** Здесь показаны те же реальные данные; высота «труб» строго равна частоте интервала, а декоративные детали не меняют значения. Пунктир обозначает медиану. [К исходным величинам](#scrollTo=4edb5f0b).

In [ ]:
# @title X · промышленная гистограмма { display-mode: "form" }

from src.ktvd_embedded import one_distribution, show
show(one_distribution(X,"X"))

**Как читать графики X:** гистограмма показывает частоты, KDE сглаживает форму (и не является точной формулой распределения), boxplot выделяет экстремальные значения, ECDF — долю наблюдений ниже заданного уровня, Q–Q проверяет форму относительно нормального закона. Отклонение Q–Q от прямой не следует автоматически называть ошибками измерений.

### Как читать графики X

1. **Гистограмма и KDE.** Распределение правосторонне асимметрично, заметна неоднородность. Высота столбца отражает частоту выбранного интервала, а KDE — сглаженную форму, зависящую от ширины окна.
2. **Boxplot.** Центральные 50% измерений находятся между квартилями Q₁ и Q₃. Точки за условными границами 1,5 IQR — кандидаты для проверки, не обязательно ошибки.
3. **ECDF.** Эмпирическая функция показывает, какая доля наблюдений не превосходит выбранное значение X.
4. **Q–Q.** Отклонения от опорной прямой указывают на ограничения нормальной модели, особенно в хвостах.

**Что проверяем дальше:** зависит ли форма распределения от концентрации и экспериментальной партии. Разные условия измерения могут создавать несколько скоплений значений, даже если каждая отдельная группа выглядит проще.


Посмотрим как зависит концентрация газа на показания сенсора:

In [ ]:
# @title X · отклик от концентрации { display-mode: "form" }
eda_conc = sample[
    ["concentration", "feature_1"]
].copy()

eda_conc["concentration"] = pd.to_numeric(
    eda_conc["concentration"],
    errors="coerce"
)

eda_conc = eda_conc.dropna()
from src.ktvd_embedded import concentration_scatter, show
show(concentration_scatter(sample,"X"))

Диаграмма концентрации помогает объяснить неоднородность распределения X. Значения отклика различаются и при совпадающих концентрациях **ацетона**, следовательно, одной концентрацией вариацию полностью объяснить нельзя.

Следующая проверка — сравнение экспериментальных партий. Связь с партиями следует интерпретировать как статистическую, а не как доказанное влияние конкретного механизма старения сенсора.


**Попробуем посмотреть сильно ли различаются показания в разных экспериментальных группах**

In [ ]:
# @title X · распределения по партиям { display-mode: "form" }

from src.ktvd_embedded import batch_box, show
show(batch_box(sample,"X"))

По этой диаграмме распределения мы видим почему в самой первой гистограмме по этой величине у нас были такие странные пики и волны. **Разные партии имеют разные характерные значения отклика нашего сенсора**.


---

Теперь сравним значений из разныз партий для одной концентрации

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
check = sample[
    ["batch", "concentration", "feature_1"]
].copy()

check["concentration"] = pd.to_numeric(
    check["concentration"],
    errors="coerce"
)

common_conc = (
    check.groupby("concentration")
    .agg(
        наблюдений=("feature_1", "count"),
        партий=("batch", "nunique")
    )
    .query("партий >= 2")
    .sort_values(
        ["партий", "наблюдений"],
        ascending=False
    )
)

display(common_conc.head(15))

Мы выяснили какой концентрации больше всего во всех частях датасета. Теперь посмотрим их зависимости

In [ ]:
# @title X · партии при концентрации 100 ppmv { display-mode: "form" }
data_100 = sample.copy()

data_100["concentration"] = pd.to_numeric(
    data_100["concentration"],
    errors="coerce"
)


data_100 = data_100[
    data_100["concentration"] == 100
].copy()


batch_stats = (
    data_100.groupby("batch")["feature_1"]
    .agg(
        Количество="count",
        Среднее="mean",
        Медиана="median",
        Стандартное_отклонение="std"
    )
    .round(2)
)

display(batch_stats)
from src.ktvd_embedded import batch_box, show
show(batch_box(sample,"X",only100=True))

**Сравнение при фиксированной концентрации:** сравнение разных партий только при концентрации 100 ppmv уменьшает влияние концентрации, но не устраняет все возможные экспериментальные факторы. Наблюдения по партиям не обязательно равновелики, поэтому визуальная разница сама по себе не доказывает причинный эффект дрейфа.

При фиксированной концентрации распределения откликов по партиям могут различаться. Это говорит о том, что помимо концентрации на показания связаны и другие условия эксперимента.

Сравнение партий нельзя трактовать как строго контролируемый причинный эксперимент: партии различаются не только временем, но и составом измерений.


**Перейдем к нормализации **

In [ ]:
# @title X · стандартизация { display-mode: "form" }
X_standardized = (X - X.mean()) / X.std(ddof=1)

print("Среднее:", round(X_standardized.mean(), 6))
print("Стандартное отклонение:",
      round(X_standardized.std(ddof=1), 6))
from src.ktvd_embedded import standardized_panel, show
show(standardized_panel(X,"X"))

Для приведения значений случайной величины к единому масштабу была применена нормализация/стандартизация z-score. Каждое значение  преобразуется путем вычитания выборочного среднего и деление на выборочное стандартное отклонение.

**Исследуем теоретическое распределение. **

Для оценки возможности описания случайной величины X теоретическим законом распределения было проведено сравнение эмпирической плотности с плотностью нормального распределения.

Параметры нормальной модели оценивались по экспериментальной выборке: математическое ожидание — выборочным средним, стандартное отклонение — соответствующей выборочной оценкой.

В качестве начальной модели выбрано нормальное распределение, поскольку оно позволяет оценить отклонения экспериментальных данных от симметричного распределения и служит базой для дальнейшего сравнения альтернативных моделей.

In [ ]:
# @title X · проверка нормальной модели { display-mode: "form" }
mu = X.mean()
sigma = X.std(ddof=1)

# Значения для теоретической кривой
x_grid = np.linspace(X.min(), X.max(), 1000)

normal_pdf = stats.norm.pdf(
    x_grid,
    loc=mu,
    scale=sigma
)
from src.ktvd_embedded import model_panel, show
show(model_panel(X,"X",only_normal=True))

Симметричная нормальная модель не отражает всей структуры эмпирического распределения X: положительная асимметрия и неоднородность остаются заметными. Более обоснованный выбор распределения требует не только визуального сравнения, но и количественных критериев и диагностики остатков.


В связи с тем, что нормальное симметричное распределение совершенно не соответствует экспериментальным данным, можно проверить асимметричное распределение.

In [ ]:
# @title X · сравнение теоретических моделей { display-mode: "form" }
x_values = X.dropna().to_numpy()

# Подбираем параметры распределений
normal_params = stats.norm.fit(x_values)
skew_params = stats.skewnorm.fit(x_values)

# Сетка значений
x_grid = np.linspace(
    x_values.min(),
    x_values.max(),
    1000
)

# Плотности
normal_pdf = stats.norm.pdf(x_grid, *normal_params)
skew_pdf = stats.skewnorm.pdf(x_grid, *skew_params)
print("Параметры skew-normal:",skew_params)
from src.ktvd_embedded import model_panel, show
show(model_panel(X,"X"))

**Ограничение модели:** форма «skew-normal» может быть визуально ближе к гистограмме, но не охватывает все возможные величины асимметрии. Нужны количественные показатели (AIC/BIC и графики остатков), а сравнение на тех же данных, по которым оценены параметры, остаётся описательным.

Асимметричное нормальное семейство способно описать правый хвост лучше, чем симметричная модель, однако ни одна из двух кривых не обязана воспроизводить все скопления наблюдений. Количественное сравнение AIC/BIC приведено в дополнительном анализе; модели оцениваются на тех же данных и не считаются подтверждёнными физическими законами.


### Independence & Estimations

Чтобы исследовать зависимости и независимости величины, начнем с того, что посмотрим на зависимости всех трёх случаныйных величин. Для следующих случайных величин такое делать не будем, сделаем такое один раз здесь. Для этого пстроим 3 диаграммы рассеяния и матрица корреляции.

In [ ]:
# @title X · Y · Z: корреляции и диаграммы рассеяния { display-mode: "form" }
sensor_data = sample[["feature_1","feature_13","feature_24"]].dropna().copy()
sensor_data.columns = ["X","Y","Z"]
corr_matrix = sensor_data.corr()
display(corr_matrix.round(3))
from src.ktvd_embedded import corr_panel, relations_panel, show
show(corr_panel(sample))
from src.ktvd_embedded import relations_panel, show
show(relations_panel(sample))

**Проверка зависимости:** ненулевой коэффициент корреляции — свидетельство линейной статистической связи, но нулевая корреляция не доказывает независимость. Если признаки измерены на одних и тех же физических записях, их независимость нельзя предполагать автоматически.

### Связи между X, Y и Z

Корреляции Пирсона, рассчитанные на фиксированной выборке из 2 500 строк, отражают разные типы совместной изменчивости:

- X–Y — положительная линейная связь;
- X–Z — сравнительно слабая отрицательная связь;
- Y–Z — наиболее выраженная среди этих трёх пар отрицательная связь.

Все коэффициенты и p-value вычисляются в предыдущей ячейке. Корреляция не равна независимости и не показывает причину: концентрация и экспериментальная партия могут одновременно влиять на несколько признаков.


In [ ]:
# @title Матрица 128 сенсорных характеристик { display-mode: "form" }
features = [f"feature_{i}" for i in range(1,129)]
corr_all=sample[features].corr(method="pearson")
from src.ktvd_embedded import corr_all_panel, show
show(corr_all_panel(corr_all))

Также также рассчитана корреляционная матрица по всей выборке.

По этой матрице хорошо видна блочная структура, обусловленная таким набором данных(16 сенсоров по 8 характеристик).


Выявлены области как положительных, так и отрицательных корреляций. Повторение цветовых структур указывает на сходные закономерности взаимосвязей между отдельными характеристиками разных сенсоров.

Таким образом, признаки сенсорного массива не являются полностью независимыми по линейным связям и могут содержать частично пересекающуюся информацию.

**Дальше посмотрим зависимость размера выборки на свойства EDA и не только**

Чтобы это выяснить, рассмотрим выборки разного объёма и проследим, как меняются оценки математического ожидания и дисперсии при увеличении числа наблюдений.

Также сравним два способа формирования выборки: с возвращением и без возвращения. Это позволит оценить, насколько способ отбора влияет на полученные результаты.

In [ ]:
# @title X · 300 повторений · состоятельность оценок { display-mode: "form" }
# Полная доступная совокупность измерений ацетона
population_X = pd.to_numeric(
    general.loc[general["gas_id"] == 5, "feature_1"],
    errors="coerce"
).dropna().to_numpy()

# Основная выборка
x = X.dropna().to_numpy()

# Параметры конечной совокупности
mu_pop = population_X.mean()
var_pop = population_X.var(ddof=0)

print(f"Размер совокупности: {len(population_X)}")
print(f"Среднее совокупности: {mu_pop:,.2f}")
print(f"Дисперсия совокупности: {var_pop:,.2f}")

# Размеры выборок
sizes = np.array([20, 50, 100, 200, 500, 1000, 1500, 2000, 2500])

# Повторяем отбор 300 раз для каждого размера
repeats = 300
rng = np.random.default_rng(42)

results = []

for n in sizes:
    for replace in [False, True]:
        means = []
        variances = []

        for _ in range(repeats):
            selected = rng.choice(
                population_X,
                size=n,
                replace=replace
            )

            means.append(selected.mean())
            variances.append(selected.var(ddof=1))

        results.append({
            "n": n,
            "Метод": "С возвращением" if replace else "Без возвращения",
            "Среднее оценок": np.mean(means),
            "SD среднего": np.std(means, ddof=1),
            "Средняя дисперсия": np.mean(variances),
            "SD дисперсии": np.std(variances, ddof=1)
        })

results_df = pd.DataFrame(results)

display(results_df.round(2))


# Исправленная дисперсия конечной совокупности
var_ref = population_X.var(ddof=1)
from src.ktvd_embedded import estimation_panel, show
show(estimation_panel(results_df,mu_pop,var_ref,"X"))
N = len(population_X)
n = len(x)

mean_x = x.mean()
std_x = x.std(ddof=1)

# Стандартная ошибка с поправкой
se = (std_x / np.sqrt(n)) * np.sqrt((N - n) / N)

# Приближённый 95% доверительный интервал
critical = stats.t.ppf(0.975, df=n - 1)

ci_lower = mean_x - critical * se
ci_upper = mean_x + critical * se

print(f"Выборочное среднее: {mean_x:,.2f}")
print(f"Стандартная ошибка: {se:,.2f}")
print(f"95% доверительный интервал: [{ci_lower:,.2f}; {ci_upper:,.2f}]")

**Устойчивость оценок:** на фиксированной конечной совокупности видно уменьшение вариации оценок среднего при увеличении n. В классическом смысле состоятельность — свойство последовательности оценок при увеличении размера выборки в вероятностной модели. Повторные выборки не заменяют доказательства, но показывают ожидаемую тенденцию. Для выборки без возвращения стандартная ошибка учитывает поправку на конечную совокупность.

### Вывод о выборочных оценках X

Для каждого размера выборки выполнено 300 повторений с возвращением и без возвращения. С увеличением n разброс оценок среднего сокращается; при отборе без возвращения заметно влияние поправки на конечную совокупность.

Доверительный интервал для среднего и его границы вычисляются выше из фактических данных и предпосылок схемы отбора. Он характеризует точность оценивания среднего **фиксированной совокупности из 3 009 строк**, а не автоматически всех будущих измерений.


## 02 / Величина Y — нарастание отклика

Центральная тенденция, асимметрия, стандартизация и выборочные оценки.


## Random Variable 02 — Y

### Description

В качестве второй случайной величины выбрана характеристика нарастания отклика уже второго газового сенсора.

В отличие от первой величины, эта отражает не амплитуду изменения, а особенности сигнала сенсора на этапе его нарастания.

Случайной величиной (Y) будем считать значение данного признака для случайно выбранного измерения ацетона из исследуемой совокупности. Изменчивость значений может быть связана с концентрацией газа, экспериментальной партией и особенностями отклика сенсора.


---

**Стохастическая природа величины**

В рамках исследования случайный эксперимент — это равновероятный выбор одного наблюдения из экспериментальной выборки.

В нашем случае Y характеризует нарастание отклика второго сенсора, рассчитанное методом экспоненциального скользящего среднего (EMAi, α = 0,1).

До выбора наблюдения значение Y неизвестно, поскольку измерения проводились при разных концентрациях ацетона, в различных экспериментальных партиях и условиях.

Следовательно, Y является случайной величиной относительно данного эксперимента. Однако с физико-химической точки зрения её значения могут подчиняться определённым закономерностям.

### EDA & Distribution

Для первоначального анализа случайной величины Y рассчитаем основные описательные статистики: выборочное среднее, медиану, дисперсию, стандартное отклонение, квартили, асимметрию и эксцесс.

Это позволит оценить характерные значения отклика второго сенсора, степень их разброса и предварительно определить особенности распределения.

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
Y = sample["feature_13"].dropna().copy()

stats_Y = pd.Series({
    "Количество наблюдений": Y.count(),
    "Среднее": Y.mean(),
    "Медиана": Y.median(),
    "Дисперсия": Y.var(ddof=1),
    "Стандартное отклонение": Y.std(ddof=1),
    "Минимум": Y.min(),
    "Q1 (25%)": Y.quantile(0.25),
    "Q3 (75%)": Y.quantile(0.75),
    "IQR": Y.quantile(0.75) - Y.quantile(0.25),
    "Максимум": Y.max(),
    "Асимметрия": Y.skew(),
    "Эксцесс": Y.kurt()
})

display(
    stats_Y.to_frame(name="Значение").style.format(
        "{:,.2f}",
        thousands=" ",
        decimal=","
    )
)

### Интерпретация статистик Y

У величины Y заметная положительная асимметрия и высокий эксцесс. Среднее, медиана и показатели разброса отличаются от X, поскольку это другая характеристика сенсорного отклика и другой масштаб измерений.

Крупные значения правого хвоста могут сильно влиять на среднее и дисперсию. Проверим этот вывод четырьмя взаимодополняющими графиками EDA; выбросы не удаляем без обоснования.


In [ ]:
# @title Y · EDA: распределение, boxplot, ECDF, Q–Q { display-mode: "form" }
from IPython.display import HTML, display
from src.ktvd_embedded import metric_cards
display(HTML(metric_cards(Y,"Y")))
from src.ktvd_embedded import eda_panel, show
show(eda_panel(Y,"Y"))

**Интерактивная гистограмма Y.** Здесь показаны те же реальные данные; высота «труб» строго равна частоте интервала, а декоративные детали не меняют значения. Пунктир обозначает медиану. [К исходным величинам](#scrollTo=b6f3df63).

In [ ]:
# @title Y · промышленная гистограмма { display-mode: "form" }

from src.ktvd_embedded import one_distribution, show
show(one_distribution(Y,"Y"))

### Интерпретация EDA для Y

1. **Гистограмма и KDE.** Распределение заметно смещено вправо. Форма сглаженной кривой зависит от выбранного окна и не подтверждает автоматически наличие отдельных субпопуляций.
2. **Boxplot.** Видны точки за условными IQR-границами; они требуют дополнительного исследования.
3. **ECDF.** Показывает долю сравнительно небольших откликов и длину правого хвоста.
4. **Q–Q.** Существенные отклонения от нормальной прямой, особенно справа, согласуются с высокой асимметрией.

Эти графики дополняют таблицу характеристик: форма распределения требует большего, чем одна средняя величина.


Для величины X отдельно рассмотрено влияние концентрации и партии. Для Y основное сравнение проведём по форме распределения и выборочным оценкам, а также учтём Y в совместном анализе зависимостей X, Y, Z.


In [ ]:
# @title Y · теоретические распределения { display-mode: "form" }
y_values = Y.dropna().to_numpy()

normal_params_Y = stats.norm.fit(y_values)
skew_params_Y = stats.skewnorm.fit(y_values)


y_grid = np.linspace(y_values.min(), y_values.max(), 1000)

normal_pdf_Y = stats.norm.pdf(y_grid, *normal_params_Y)
skew_pdf_Y = stats.skewnorm.pdf(y_grid, *skew_params_Y)
from src.ktvd_embedded import model_panel, show
show(model_panel(Y,"Y"))

Асимметричная нормальная кривая визуально лучше учитывает правый хвост Y, однако не гарантирует совпадения распределений. Для вывода о качестве приближения используем сравнительные метрики AIC/BIC и диагностику, а не только внешний вид кривых.


Как и для X проведем стандартизацию случайной величины Y



In [ ]:
# @title Y · стандартизация { display-mode: "form" }
# Стандартизация Y
Y_standardized = (Y - Y.mean()) / Y.std(ddof=1)

# Проверка
print("Среднее:", round(Y_standardized.mean(), 6))
print(
    "Стандартное отклонение:",
    round(Y_standardized.std(ddof=1), 6)
)

# Сравнение распределений
from src.ktvd_embedded import standardized_panel, show
show(standardized_panel(Y,"Y"))

Опять же, форма графика и его характерные пики не поменялись, поменялся только масштаб.

Таким образом, стандартизация позволяет сравнивать характеристики сенсорных откликов, измеряемые в разных масштабах, но не делает распределение нормальным.

### Independence & Estimations

### Сходимость оценок Y

По всем 3 009 наблюдениям класса 5 (ацетон) вычисляются параметры конечной совокупности. Затем для нескольких значений n проводится по 300 повторных выборок **с возвращением и без возвращения**.

Сопоставление выборочных средних и дисперсий с эталонами полной таблицы позволяет оценить влияние объёма и способа отбора. Предположения о новых физических опытах из этой симуляции не следуют.


In [ ]:
# @title Y · 300 повторений · состоятельность оценок { display-mode: "form" }
population_Y = pd.to_numeric(
    general.loc[general["gas_id"] == 5, "feature_13"],
    errors="coerce"
).dropna().to_numpy()

y = Y.dropna().to_numpy()

mu_pop_Y = population_Y.mean()
var_pop_Y = population_Y.var(ddof=1)

# Параметры моделирования
sizes = [20, 50, 100, 200, 500, 1000, 1500, 2000, 2500]
repeats = 300
rng = np.random.default_rng(42)

results_Y = []

for n in sizes:
    for replace in [False, True]:

        means = []
        variances = []

        for _ in range(repeats):
            selected = rng.choice(
                population_Y,
                size=n,
                replace=replace
            )

            means.append(selected.mean())
            variances.append(selected.var(ddof=1))

        results_Y.append({
            "n": n,
            "Метод": "С возвращением" if replace else "Без возвращения",
            "Среднее оценок": np.mean(means),
            "SD среднего": np.std(means, ddof=1),
            "Средняя дисперсия": np.mean(variances),
            "SD дисперсии": np.std(variances, ddof=1)
        })

results_Y_df = pd.DataFrame(results_Y)

display(results_Y_df.round(2))
from src.ktvd_embedded import estimation_panel, show
show(estimation_panel(results_Y_df,mu_pop_Y,var_pop_Y,"Y"))
N = len(population_Y)
n = len(y)

mean_y = y.mean()
std_y = y.std(ddof=1)

# Стандартная ошибка с поправкой на конечную совокупность
se_y = (
    std_y / np.sqrt(n)
    * np.sqrt((N - n) / N)
)

critical = stats.t.ppf(0.975, df=n - 1)

ci_lower_y = mean_y - critical * se_y
ci_upper_y = mean_y + critical * se_y

print(f"Выборочное среднее: {mean_y:,.2f}")
print(f"Стандартная ошибка: {se_y:,.2f}")
print(
    f"95% доверительный интервал: "
    f"[{ci_lower_y:,.2f}; {ci_upper_y:,.2f}]"
)

**Устойчивость оценок:** на фиксированной конечной совокупности видно уменьшение вариации оценок среднего при увеличении n. В классическом смысле состоятельность — свойство последовательности оценок при увеличении размера выборки в вероятностной модели. Повторные выборки не заменяют доказательства, но показывают ожидаемую тенденцию. Для выборки без возвращения стандартная ошибка учитывает поправку на конечную совокупность.

### Результат моделирования оценок Y

С ростом размера выборки средние оценки остаются близкими к характеристикам доступной совокупности, а разброс оценок обычно уменьшается. Для больших n отбор без возвращения даёт меньшую вариацию среднего, чем отбор с возвращением. Точные величины разброса приведены в рассчитанной выше таблице, а не зафиксированы вручную в тексте.


## 03 / Величина Z — спад отклика

Отрицательные значения, распределение и оценивание.


## Random Variable 03 — Z

### Description

В качестве третьей случайной величины была выбрана характеристика спада отклика уже на третьем сенсоре, рассчитанного тем же способом, что и величина Y, поэтому возможно, результаты анализа будут в чем-то схожи с предыдущей величиной.



---

**Стохастическая природа**

В рамках исследования случайный эксперимент — это равновероятный выбор одного наблюдения из экспериментальной выборки.

Значение Z заранее неизвестно, поскольку измерения проводились при разных концентрациях ацетона, в различных экспериментальных партиях и условиях.

Следовательно, Z является случайной величиной относительно данного эксперимента. При этом её значения могут зависеть от физико-химических характеристик сенсора и условий измерения.

### EDA & Distribution

Рассчитываем все характеристики

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
Z = sample["feature_24"].dropna().copy()

stats_Z = pd.Series({
    "Количество наблюдений": Z.count(),
    "Среднее": Z.mean(),
    "Медиана": Z.median(),
    "Дисперсия": Z.var(ddof=1),
    "Стандартное отклонение": Z.std(ddof=1),
    "Минимум": Z.min(),
    "Q1 (25%)": Z.quantile(0.25),
    "Q3 (75%)": Z.quantile(0.75),
    "IQR": Z.quantile(0.75) - Z.quantile(0.25),
    "Максимум": Z.max(),
    "Асимметрия": Z.skew(),
    "Эксцесс": Z.kurt()
})

display(
    stats_Z.to_frame(name="Значение").style.format(
        "{:,.2f}",
        thousands=" ",
        decimal=","
    )
)

### Интерпретация характеристик Z

Значения Z преимущественно отрицательны, в отличие от X и Y. Выраженная левосторонняя асимметрия и высокий положительный эксцесс указывают на длинный отрицательный хвост и крайние наблюдения.

Это особенности статистического распределения признака `feature_24`; они не доказывают отдельно механизм изменения сопротивления сенсора.


Также посмотрим на это все в формате графиков

In [ ]:
# @title Z · EDA: распределение, boxplot, ECDF, Q–Q { display-mode: "form" }
from IPython.display import HTML, display
from src.ktvd_embedded import metric_cards
display(HTML(metric_cards(Z,"Z")))
from src.ktvd_embedded import eda_panel, show
show(eda_panel(Z,"Z"))

**Интерактивная гистограмма Z.** Здесь показаны те же реальные данные; высота «труб» строго равна частоте интервала, а декоративные детали не меняют значения. Пунктир обозначает медиану. [К исходным величинам](#scrollTo=5f3e90a9).

In [ ]:
# @title Z · промышленная гистограмма { display-mode: "form" }

from src.ktvd_embedded import one_distribution, show
show(one_distribution(Z,"Z"))

### Интерпретация EDA для Z

1. **Гистограмма и KDE.** Большинство значений сосредоточено ближе к нулю, тогда как отрицательный хвост заметно длиннее.
2. **Boxplot.** Слева видны потенциальные IQR-выбросы; это не тождественно ошибкам измерения.
3. **ECDF.** Позволяет прочитать накопленную долю наблюдений при отрицательных порогах.
4. **Q–Q.** Существенные отклонения в левом хвосте показывают ограниченность нормальной модели.

Комбинация графиков позволяет отделить центр, разброс, асимметрию и экстремальные значения.


**Сравнение симметричной нормальной и асимметричной нормальной моделей**

In [ ]:
# @title Z · теоретические распределения { display-mode: "form" }
z_values = Z.dropna().to_numpy()

normal_params_Z = stats.norm.fit(z_values)
skew_params_Z = stats.skewnorm.fit(z_values)

z_grid = np.linspace(z_values.min(), z_values.max(), 1000)

normal_pdf_Z = stats.norm.pdf(z_grid, *normal_params_Z)
skew_pdf_Z = stats.skewnorm.pdf(z_grid, *skew_params_Z)
from src.ktvd_embedded import model_panel, show
show(model_panel(Z,"Z"))

Для Z асимметричная модель может лучше учитывать левый хвост, чем симметричная нормальная, но даже она не обязана воспроизводить все крайние значения. Сопоставление по AIC/BIC приведено ниже в дополнительных исследованиях.


Перейдем к стандартизации.

In [ ]:
# @title Z · стандартизация { display-mode: "form" }
# Стандартизация третьей случайной величины
Z_standardized = (Z - Z.mean()) / Z.std(ddof=1)

# Проверяем результат
print("Среднее:", round(Z_standardized.mean(), 6))
print(
    "Стандартное отклонение:",
    round(Z_standardized.std(ddof=1), 6)
)

# Сравниваем распределения
from src.ktvd_embedded import standardized_panel, show
show(standardized_panel(Z,"Z"))

Для приведения значений случайной величины Z к единому масштабу была выполнена стандартизация методом Z-score.

После преобразования выборочное среднее стало равным нулю, а стандартное отклонение — единице. При этом форма распределения сохранилась, включая выраженную левостороннюю асимметрию и наличие крайних отрицательных значений.

### Independence & Estimations

Для исследования устойчивости оценок математического ожидания и дисперсии случайной величины Z рассмотрим выборки различного объёма.

Для каждого объёма проведём по 300 случайных отборов с возвращением и без возвращения. Полученные оценки сравним с характеристиками всех доступных измерений ацетона.

Это позволит оценить, насколько увеличение числа наблюдений и выбранный способ отбора влияют на точность и устойчивость статистических оценок.

In [ ]:
# @title Z · 300 повторений · состоятельность оценок { display-mode: "form" }
population_Z = pd.to_numeric(
    general.loc[general["gas_id"] == 5, "feature_24"],
    errors="coerce"
).dropna().to_numpy()
z = Z.dropna().to_numpy()

mu_pop_Z = population_Z.mean()
var_pop_Z = population_Z.var(ddof=1)

# Параметры эксперимента
sizes = [20, 50, 100, 200, 500, 1000, 1500, 2000, 2500]
repeats = 300
rng = np.random.default_rng(42)

results_Z = []

for n in sizes:
    for replace in [False, True]:

        means = []
        variances = []

        for _ in range(repeats):
            selected = rng.choice(
                population_Z,
                size=n,
                replace=replace
            )

            means.append(selected.mean())
            variances.append(selected.var(ddof=1))

        results_Z.append({
            "n": n,
            "Метод": "С возвращением" if replace else "Без возвращения",
            "Среднее оценок": np.mean(means),
            "SD среднего": np.std(means, ddof=1),
            "Средняя дисперсия": np.mean(variances),
            "SD дисперсии": np.std(variances, ddof=1)
        })

results_Z_df = pd.DataFrame(results_Z)

display(results_Z_df.round(2))
from src.ktvd_embedded import estimation_panel, show
show(estimation_panel(results_Z_df,mu_pop_Z,var_pop_Z,"Z"))

**Устойчивость оценок:** на фиксированной конечной совокупности видно уменьшение вариации оценок среднего при увеличении n. В классическом смысле состоятельность — свойство последовательности оценок при увеличении размера выборки в вероятностной модели. Повторные выборки не заменяют доказательства, но показывают ожидаемую тенденцию. Для выборки без возвращения стандартная ошибка учитывает поправку на конечную совокупность.

### Результат моделирования оценок Z

При увеличении n колебания оценок среднего и дисперсии вокруг параметров фиксированной совокупности в целом снижаются. Особенно это заметно при отборе без возвращения: доступная совокупность конечна, поэтому по мере приближения n к N неопределённость среднего сокращается.

Для интерпретации используйте рассчитанные в предыдущей ячейке значения, поскольку точные результаты зависят от фиксированного `seed=42` и самой выборки.


## Lab resume

### Проверка статистической гипотезы Y–Z

Проверим наличие линейной связи между признаками Y и Z на выбранных 2 500 наблюдениях.

$$H_0:\rho_{YZ}=0,\qquad H_1:\rho_{YZ}\ne0,\qquad \alpha=0.05.$$

Используется двусторонняя проверка корреляции Пирсона. Её p-value интерпретируется при предположении о независимых наблюдениях и применимости теста; наличие общих партий и концентраций может нарушать эти предпосылки. После основной проверки дополнительно исследуется связь с учётом экспериментальных условий.


In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
hyp_data = sample[
    ["feature_13", "feature_24"]
].dropna()

Y_test = hyp_data["feature_13"]
Z_test = hyp_data["feature_24"]

# Проверка гипотезы
r, p_value = stats.pearsonr(Y_test, Z_test)

alpha = 0.05

print(f"Количество наблюдений: {len(hyp_data)}")
print(f"Коэффициент корреляции: {r:.4f}")
print(f"p-value: {p_value:.3e}")
print(f"Уровень значимости: {alpha}")

if p_value < alpha:
    print("H0 отвергается: корреляция статистически значима.")
else:
    print("Нет оснований отвергнуть H0.")

**Проверяемая гипотеза:** гипотеза в коде касается пары **Y–Z**, а не X–Z. При небольшом p-value отвергается H₀: нулевая корреляция, но причинность не установлена. Общие факторы (концентрация, партия) могут порождать существенную часть корреляции.

**Результат теста (Y–Z).** Для 2500 наблюдений найден отрицательный коэффициент корреляции Пирсона. Значение получено непосредственно в предыдущей ячейке и зависит от фиксированного случайного отбора (`seed=42`). При p-value ниже 0,05 отвергаем H₀ о нулевой линейной корреляции. Это не доказывает причинность: на обе характеристики могут одновременно влиять концентрация и экспериментальная партия. Дополнительную проверку после учёта этих факторов см. в разделе «Гипотезы и стратификация».

---

### Выводы первого спринта

В исходном наборе 13 910 экспериментальных записей газовых сенсоров. Для подробного анализа выбран **класс 5 — ацетон (3 009 наблюдений)**, из которого воспроизводимо отобраны **2 500** наблюдений без возвращения.

Для X (`feature_1`), Y (`feature_13`), Z (`feature_24`) рассмотрены описательные статистики, распределения и их диагностика, стандартизация, связи, влияние концентраций и партий, а также сходимость выборочных оценок при разных способах отбора.

Наблюдения неоднородны; простая общая нормальная модель может недостаточно описывать отдельные партии. IQR-флаги сами по себе не доказывают ошибочность измерения. Обнаруженная зависимость Y–Z — статистическое описание, а не установленная причинная связь. Интерпретация относится к наблюдаемой конечной совокупности.


**Вывод по результатам первого спринта.** Исследование охватило больше, чем минимальный EDA: изучены особенности разметки, выборка из конечной совокупности, три распределения, влияние условий и повторные случайные отборы. Дополнительные модули ниже дополняют авторский анализ математическими пояснениями, количественной проверкой моделей, диагностикой выбросов и генераторами. Исходные заметки автора доступны без изменений в `notebooks/original/LR_1_stat-2_UNCHANGED.ipynb`.

### Критерии первого спринта

Подтверждения по структуре, случайным величинам X/Y/Z, EDA, оценкам, математической базе и графикам сведены в [итоговую таблицу критериев](#scrollTo=0e7069b9). Там же приведены дополнительные результаты Diamonds и ссылки на соответствующие разделы.


### Литература



1. Vergara A. *Gas Sensor Array Drift at Different Concentrations* [Электронный ресурс] // UCI Machine Learning Repository. — 2012. — DOI: [10.24432/C5MK6M](https://doi.org/10.24432/C5MK6M).

2. Vergara A., Vembu S., Ayhan T., Ryan M. A., Homer M. L., Huerta R. Chemical gas sensor drift compensation using classifier ensembles // Sensors and Actuators B: Chemical. — 2012. — Vol. 166–167. — P. 320–329. — DOI: [10.1016/j.snb.2012.01.074](https://doi.org/10.1016/j.snb.2012.01.074).

3. Rodríguez-Luján I., Fonollosa J., Vergara A., Homer M., Huerta R. On the calibration of sensor arrays for pattern recognition using the minimal number of experiments // Chemometrics and Intelligent Laboratory Systems. — 2014. — Vol. 130. — P. 123–134. — DOI: [10.1016/j.chemolab.2013.10.012](https://doi.org/10.1016/j.chemolab.2013.10.012).

4. Fonollosa J., Rodríguez-Luján I., Huerta R. Chemical gas sensor array dataset // Data in Brief. — 2015. — Vol. 3. — P. 85–89. — DOI: [10.1016/j.dib.2015.01.003](https://doi.org/10.1016/j.dib.2015.01.003).

5. Heckert N. A., Filliben J. J. NIST/SEMATECH e-Handbook of Statistical Methods. Chapter 1: Exploratory Data Analysis [Электронный ресурс]. — National Institute of Standards and Technology, 2003. — URL: https://www.nist.gov/publications/nistsematech-e-handbook-statistical-methods-chapter-1-exploratory-data-analysis.

6. SciPy Developers. *SciPy Statistical Functions* [Электронный ресурс]. — Официальная документация библиотеки SciPy. — URL: https://docs.scipy.org/doc/scipy/reference/stats.html.

7. pandas Development Team. *pandas.DataFrame.corr* [Электронный ресурс]. — Официальная документация pandas. — URL: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html.

## Расширенная математическая основа и соглашения

Для каждого признака X, Y, Z наблюдаем случайный отклик при неполностью
контролируемой совокупности факторов: микросостояние сенсора, концентрация,
износ, условия измерения и шум. **Величина случайна в контексте повторного
эксперимента**; сама запись файла, конечно, уже фиксирована.

Оценки при независимом одинаково распределённом отборе:

$$\overline X_n=\frac1n\sum_{i=1}^n X_i,\quad
S_n^2=\frac1{n-1}\sum_{i=1}^n(X_i-\overline X_n)^2.$$

Стандартизация (не означает нормальность):

$$Z_i=\frac{X_i-\overline X}{S_n},\qquad
\overline Z\approx0,\quad S_Z\approx1.$$

При простой случайной выборке **без возвращения** из конечной совокупности размера N:

$$\operatorname{Var}(\overline X_n)=\frac{S_N^2}{n}\frac{N-n}{N},\qquad
SE(\overline X_n)\approx\frac{s}{\sqrt n}\sqrt{\frac{N-n}{N}},$$

где $S_N^2=\frac1{N-1}\sum_{i=1}^N(x_i-\overline x_N)^2$.
Последняя формула использует выборочную оценку $s^2$ для конечной генеральной дисперсии $S_N^2$ (с поправкой $N-1$). Доверительные интервалы опираются на
предпосылки отбора, а не только на форму гистограммы.

Нормальное распределение:
$$f(x;\mu,\sigma)=\frac1{\sigma\sqrt{2\pi}}
\exp\bigl[-(x-\mu)^2/(2\sigma^2)\bigr],\quad\sigma>0.$$

Асимметричное нормальное:
$$f(x;\xi,\omega,\alpha)=\frac2\omega
\phi\left(\frac{x-\xi}{\omega}\right)
\Phi\left(\alpha\frac{x-\xi}{\omega}\right),\quad\omega>0.$$

Оценку качества подгонки по AIC используем **только как сравнение выбранных моделей**:
$$\mathrm{AIC}=2k-2\log\widehat L,\qquad
\mathrm{BIC}=k\log n-2\log\widehat L.$$
Меньшее значение — относительное предпочтение при данных моделях и их
предпосылках; это не сертификат «правильного» распределения.

## Дополнительное исследование: количественное качество теоретических моделей

Сравниваем нормальное и skew-normal семейства для **всех трёх** признаков.
Параметры подбираются максимумом правдоподобия на той же выборке; AIC/BIC — описательные показатели,
а не независимая проверка на новых экспериментах. Результаты дополняют (не заменяют) все исходные графики автора.

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
fits=[]
for label,feature in {'X':'feature_1','Y':'feature_13','Z':'feature_24'}.items():
    fits.extend(fit_diagnostics(sample[feature].to_numpy(dtype=float),label))
fit_table=pd.DataFrame(fits)
display(fit_table.round(3))
for label in ('X','Y','Z'):
    part=fit_table.query('variable == @label').sort_values('AIC')
    print(f'{label}: минимальный AIC у модели {part.iloc[0]["model"]}; разница AIC = {part["AIC"].max()-part["AIC"].min():.2f}. Это только относительное сравнение двух семейств.')

## Дополнительное исследование: выбросы и очистка без искажения данных

Сначала выявляем IQR-флаги **как возможные аномалии**, а не автоматические ошибки.
Для каждого признака сравниваем исходную оценку и результат после фильтрации;
последний не заменяет основное исследование. Если выброс связан с реальным режимом
концентрации или партии, удаление может сделать вывод необъективным.

In [ ]:
# @title Чувствительность к потенциальным выбросам { display-mode: "form" }
impact=[]
for label,feature in {'X':'feature_1','Y':'feature_13','Z':'feature_24'}.items():
    impact.extend(outlier_impact(sample[feature].to_numpy(dtype=float),label))
impact_table=pd.DataFrame(impact)
display(impact_table.round(3))
from src.ktvd_embedded import outlier_panel, show
show(outlier_panel(impact_table))

## Дополнительная проверка: зависимость и влияние условий измерения

Тестируем пару Y–Z по полным наблюдениям и по остаткам линейной регрессии
после учёта логарифма концентрации и индикаторов партии. **Частная корреляция
не является доказательством причинности или независимости.**

$$H_0:\rho_{YZ}=0,\qquad H_1:\rho_{YZ}\ne0,\quad\alpha=0.05.$$

Используем Pearson (линейная связь) и Spearman (монотонная связь),
а также сопоставляем их с корреляцией остатков.

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
dependence_summary=residual_corr(sample)
display(dependence_summary.round(6))
from scipy import stats as spstats
for a,b in [('feature_1','feature_13'),('feature_1','feature_24'),('feature_13','feature_24')]:
    r,p=spstats.pearsonr(sample[a],sample[b]); print(f'{a} × {b}: r={r:.4f}, p={p:.2e}')

### Непараметрический бутстрэп (дополнительная иллюстрация)

Для сопоставления приводим бутстрэп-диапазоны по 2 000 повторным выборкам.
Он предполагает независимость строк, поэтому при сильной зависимости по партиям
его интервалы следует интерпретировать осторожно. Вводить бутстрэп вместо
корректного дизайна исследования нельзя.

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
rows=[]
for label,feature in {'X':'feature_1','Y':'feature_13','Z':'feature_24'}.items():
    ci=bootstrap_mean_ci(sample[feature].to_numpy(),seed=42,repeats=2000)
    rows.append({'Признак':label,'Среднее':sample[feature].mean(),
                 'Нижняя граница 95%':ci[0],'Верхняя граница 95%':ci[1]})
display(pd.DataFrame(rows).round(3))

## Интерактивный атлас исследования

Шесть панелей для самостоятельного сравнения экспериментальных данных и моделей.


### 01 / Распределения X, Y, Z

Меню над графиком переключает величины; пунктир указывает медиану.


In [ ]:
# @title ИНТЕРАКТИВ · 01 Распределения X, Y, Z { display-mode: "form" }
from src.ktvd_visuals import distribution_dashboard
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'plotly_mimetype'
chart_01 = distribution_dashboard(sample)
chart_01.show(renderer=pio.renderers.default,config={'displaylogo':False,'responsive':True,
  'modeBarButtonsToRemove':['lasso2d','select2d']})

**Интерпретация.** Высота каждой трубы в точности равна числу наблюдений в соответствующем интервале. Значения X, Y и Z имеют разные диапазоны, поэтому их оси изменяются отдельно.

[К содержанию](#scrollTo=28df4a97).


### 02 / Чувствительность к числу интервалов

Изменяйте разбиение гистограммы: форма графика зависит от ширины интервалов.


In [ ]:
# @title ИНТЕРАКТИВ · 02 Чувствительность к разбиению { display-mode: "form" }
from src.ktvd_visuals import binning_slider
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'plotly_mimetype'
chart_02 = binning_slider(sample)
chart_02.show(renderer=pio.renderers.default,config={'displaylogo':False,'responsive':True,
  'modeBarButtonsToRemove':['lasso2d','select2d']})

**Интерпретация.** Одна и та же выборка может выглядеть более гладкой или более «пиковой» при разных ширинах интервала. Поэтому число интервалов — часть методики визуализации, которую нужно указывать.

[К содержанию](#scrollTo=28df4a97).


### 03 / Сравнение экспериментальных партий

Сопоставьте партии для всех концентраций и отдельно для 100 ppmv.


In [ ]:
# @title ИНТЕРАКТИВ · 03 Измерения по партиям { display-mode: "form" }
from src.ktvd_visuals import batch_explorer
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'plotly_mimetype'
chart_03 = batch_explorer(sample)
chart_03.show(renderer=pio.renderers.default,config={'displaylogo':False,'responsive':True,
  'modeBarButtonsToRemove':['lasso2d','select2d']})

**Интерпретация.** Сравнение только при 100 ppmv помогает уменьшить смешение эффекта партии с эффектом концентрации. Однако партии остаются неоднородными, и сам boxplot не доказывает причину различий.

[К содержанию](#scrollTo=28df4a97).


### 04 / Зависимости X, Y и Z

Переключайте пары признаков; цвет соответствует экспериментальной партии.


In [ ]:
# @title ИНТЕРАКТИВ · 04 Связи между величинами { display-mode: "form" }
from src.ktvd_visuals import relation_explorer
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'plotly_mimetype'
chart_04 = relation_explorer(sample)
chart_04.show(renderer=pio.renderers.default,config={'displaylogo':False,'responsive':True,
  'modeBarButtonsToRemove':['lasso2d','select2d']})

**Интерпретация.** Коэффициент Пирсона характеризует линейную связь, но не причинность. Наличие общей партии и концентрации нужно учитывать при интерпретации корреляций.

[К содержанию](#scrollTo=28df4a97).


### 05 / Объём выборки и устойчивость оценок

Сравните распределения выборочных средних при разных n и способах отбора.


In [ ]:
# @title ИНТЕРАКТИВ · 05 Размер выборки и устойчивость { display-mode: "form" }
from src.ktvd_visuals import sample_size_explorer
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'plotly_mimetype'
chart_05 = sample_size_explorer(population,reps=300)
chart_05.show(renderer=pio.renderers.default,config={'displaylogo':False,'responsive':True,
  'modeBarButtonsToRemove':['lasso2d','select2d']})

**Интерпретация.** При увеличении n разброс выборочных средних обычно сокращается; при отборе без возвращения появляется поправка на конечность совокупности. Это эмпирическая демонстрация на фиксированном наборе, а не доказательство всех условий состоятельности.

[К содержанию](#scrollTo=28df4a97).


### 06 / Теоретические модели

Сравните нормальную и skew-normal модели по эмпирическим данным и AIC.


In [ ]:
# @title ИНТЕРАКТИВ · 06 Теоретические модели { display-mode: "form" }
from src.ktvd_visuals import model_explorer
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'plotly_mimetype'
chart_06 = model_explorer(sample)
chart_06.show(renderer=pio.renderers.default,config={'displaylogo':False,'responsive':True,
  'modeBarButtonsToRemove':['lasso2d','select2d']})

**Интерпретация.** Меньший AIC указывает на лучшее относительное соответствие среди сравниваемых моделей, но не означает, что модель физически правильна или точно отражает хвосты распределения.

[К содержанию](#scrollTo=28df4a97).


# Lab01 — sprint 2 · Генераторы случайных чисел

Генерация приведена как **отдельное** дополнение, не смешиваемое с исходными наблюдениями.
Прежде чем применять теорию распределений, проверяем, как меняются статистики и гистограммы.

## Математическое обоснование методов генерации

**Метод обратной функции:** если $U\sim U(0,1)$ и F непрерывна, то
$$X=F^{-1}(U)\implies\Pr(X\le x)=\Pr(U\le F(x))=F(x).$$
Для эмпирической CDF $F_n$ обобщённая обратная функция даёт выборку с возвращением
из исходных значений. Это **не новая физическая модель** датчика.

**Нормальный квантильный генератор:**
$$X=\hat\mu+\hat\sigma\Phi^{-1}(U).$$

**Skew-normal квантильный генератор:**
$$X=F_{\mathrm{SN}}^{-1}(U;\hat\xi,\hat\omega,\hat\alpha).$$

Сопоставляем как минимум две альтернативы, учитываем AIC/BIC при выборе
теоретической модели и оцениваем различия эмпирических CDF.

In [ ]:
# @title Генераторы · сравнение распределений и ECDF { display-mode: "form" }
reference=sample['feature_1'].to_numpy(dtype=float)
seed=2026
n_gen=10000
emp=empirical_quantile_generator(reference,n=n_gen,seed=seed)
rng=np.random.default_rng(seed)
u=np.clip(rng.random(n_gen),np.finfo(float).eps,1-np.finfo(float).eps)
normal=spstats.norm.ppf(u,loc=reference.mean(),scale=reference.std(ddof=1))
skew=skewnormal_generator(reference,n=n_gen,seed=seed)
model_results=[]
for label,data in [('Эмпирический квантиль',emp),('Нормальная инверсия',normal),('Skew-normal инверсия',skew)]:
    test=spstats.ks_2samp(reference,data)
    model_results.append({'Генератор':label,'Среднее':np.mean(data),'Медиана':np.median(data),
        'SD':np.std(data,ddof=1),'Q25':np.quantile(data,.25),'Q75':np.quantile(data,.75),
        'KS расстояние':test.statistic,'p (только диагностика)':test.pvalue})
generator_table=pd.DataFrame(model_results)
display(generator_table.round(4))
from src.ktvd_embedded import generator_panel, show
show(generator_panel(reference,[('Эмпирический квантиль',emp),('Нормальная инверсия',normal),('Skew-normal инверсия',skew)]))

> **Границы статистической интерпретации:** двухвыборочный KS на данных, которыми обучали эмпирический
> генератор, не является независимой оценкой качества физической модели.
> Большие n приводят к очень малым p даже для небольших отклонений;
> сравнивайте также величину расстояния, графики и практический смысл.

## Автотесты и проверка воспроизводимости

Тесты запускаются стандартным `pytest`; в репозитории находятся исходные функции,
на которых можно проверить вычисления и граничные случаи.

In [ ]:
# @title Исследовательский код · расчёты { display-mode: "form" }
import subprocess,sys
result=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests')],cwd=ROOT,
    capture_output=True,text=True,timeout=120)
print(result.stdout[-3500:])
if result.returncode:
    print(result.stderr[-1500:])
    raise AssertionError('Автотесты завершились с ошибкой')

## Дополнительные интерактивные приложения

Интерактивные графики основного анализа расположены в соответствующих ячейках и в **визуальном атласе** ниже по навигации.

Отдельные формы представления этой же работы:

- HTML-сайт с исследованием газовых сенсоров и дополнительным разделом Diamonds;
- приложение **marimo** с живыми фильтрами, пересчётом статистик и генератором.

Они дополняют ноутбук, но не заменяют его вычислительную часть. Для защиты следует открыть именно опубликованные командой версии файлов — ссылки на личные локальные пути в Colab не работают.


# Дополнительное исследование КТвД · Diamonds

**Масса и цена бриллиантов** · независимый набор данных для дизайнерского исследования.  
**Авторы дополнительного исследования:** Макарова Юлия (501809), Степанова Дарья (504599).

В этом разделе рассматриваются две случайные величины: **масса `carat` (карат)** и **цена `price` (USD)**. Сохраняются постановка задачи, EDA, математическое обоснование, доказательство зависимости в эмпирической модели, генераторы, автотесты и авторские интерактивные панели.

**Официальное описание:** [Diamonds — ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html). В документации заявлены **53 940 записей и 10 признаков**. [CSV, используемый в исследовании](https://waf.cs.illinois.edu/discovery/diamonds.csv); при недоступности этой копии применён [зеркальный CSV ggplot2 из Rdatasets](https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/ggplot2/diamonds.csv).

**Разграничение источников:** ggplot2 — официальный документированный источник набора Diamonds, **это не ResearchGate**. Источник прямо указан в разделе и в списке литературы; дополнительное исследование оценивается отдельно от газовых сенсоров.

[К содержанию всей лабораторной](#scrollTo=b0bbdf36) · [К содержанию раздела Diamonds](#scrollTo=dm-contents) · [К итоговым критериям](#scrollTo=0e7069b9)


### Постановка задачи и ограничения

### Вероятностная модель · исследование данных · генерация

Макарова Юлия 501809 · Степанова Дарья 504599

---

Цель — исследовать распределения массы и цены, проверить зависимость и сопоставить совместную и независимую генерацию. Объект анализа — записи исторического набора diamonds; результаты не описывают текущие предложения всего рынка.

[Содержание](#scrollTo=dm-contents) · [Заключение](#scrollTo=dm-section-24)

### Содержание исследования Diamonds

- [1. Загрузка и аудит](#scrollTo=dm-section-01)
- [2. Почему X и Y — случайные величины](#scrollTo=dm-section-02)
  - [Формулы характеристик](#scrollTo=dm-section-03)
- [3. Одномерный EDA](#scrollTo=dm-section-04)
  - [Гистограмма и KDE: Масса, карат](#scrollTo=dm-section-05)
  - [Boxplot: Масса, карат](#scrollTo=dm-section-06)
  - [Эмпирическая функция распределения: Масса, карат](#scrollTo=dm-section-07)
  - [Q–Q plot: Масса, карат](#scrollTo=dm-section-08)
  - [Гистограмма и KDE: Цена, USD](#scrollTo=dm-section-09)
  - [Boxplot: Цена, USD](#scrollTo=dm-section-10)
  - [Эмпирическая функция распределения: Цена, USD](#scrollTo=dm-section-11)
  - [Q–Q plot: Цена, USD](#scrollTo=dm-section-12)
  - [Проверка крайних значений](#scrollTo=dm-section-13)
- [4. Совместный EDA и группировки](#scrollTo=dm-section-14)
  - [Обсуждение совместных графиков](#scrollTo=dm-section-15)
- [5. Точное доказательство зависимости в эмпирической модели](#scrollTo=dm-section-16)
- [6. Корреляции: дополнительная характеристика связи](#scrollTo=dm-section-17)
- [7. Генератор и его математическое обоснование](#scrollTo=dm-section-18)
  - [Законы генераторов](#scrollTo=dm-section-19)
  - [Почему средних недостаточно](#scrollTo=dm-section-20)
- [8. Интерактивный генератор](#scrollTo=dm-section-21)
- [9. Контроль генератора и сходимость оценок](#scrollTo=dm-section-22)
  - [Теория ошибки среднего](#scrollTo=dm-section-23)
- [10. Итоговые выводы](#scrollTo=dm-section-24)


In [ ]:
# @title Diamonds · визуализации и управление { display-mode: "form" }
try:
    import ipywidgets
except ImportError:
    %pip -q install ipywidgets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets
from pathlib import Path
import hashlib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from src.ktvd_visuals import (
    styled as _diamonds_main_style,
    INK as DIAMONDS_INK,
    METAL as DIAMONDS_METAL,
    EDGE as DIAMONDS_EDGE,
    HIGHLIGHT as DIAMONDS_HIGHLIGHT,
    SHADOW as DIAMONDS_SHADOW,
    WINE as DIAMONDS_WINE,
    TEAL as DIAMONDS_TEAL,
    GRID as DIAMONDS_GRID,
    MUTED as DIAMONDS_MUTED,
)

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pass

DIAMONDS_SEED = 42
DIAMONDS_RESULTS = (ROOT / "results" / "diamonds") if "ROOT" in globals() else Path("results/diamonds")
DIAMONDS_RESULTS.mkdir(parents=True, exist_ok=True)
DIAMONDS_PALETTE = [DIAMONDS_METAL, DIAMONDS_EDGE, DIAMONDS_TEAL, "#8798AC", "#BCCCD6"]
DIAMONDS_LABELS = {"carat": "Масса, карат", "price": "Цена, USD"}


def diamonds_selected_data(data, mass, price, cut):
    mask = data.carat.between(*mass) & data.price.between(*price)
    if cut != "Все":
        mask &= data.cut.eq(cut)
    return data.loc[mask].copy()


def _diamonds_add_pipe(fig, edges, heights, counts, *, row=None, col=None, showlegend=False, label="Наблюдения"):
    """Металлические столбцы как в основном исследовании; числа не меняются."""
    heights = np.asarray(heights, dtype=float)
    counts = np.asarray(counts, dtype=int)
    edges = np.asarray(edges, dtype=float)
    centers = (edges[:-1] + edges[1:]) / 2
    widths = np.diff(edges)
    extra = {} if row is None else {"row": row, "col": col}
    customdata = np.column_stack([edges[:-1], edges[1:], counts])
    fig.add_trace(go.Bar(
        x=centers, y=heights, width=widths * .80,
        marker=dict(color=DIAMONDS_METAL, line=dict(color=DIAMONDS_EDGE, width=.85)),
        customdata=customdata,
        hovertemplate="Интервал: %{customdata[0]:.4g} — %{customdata[1]:.4g}<br>Наблюдений: %{customdata[2]:.0f}<extra></extra>",
        name=label, legendgroup="real", showlegend=showlegend,
    ), **extra)
    fig.add_trace(go.Bar(
        x=centers - widths * .20, y=heights, width=widths * .10,
        marker_color=DIAMONDS_HIGHLIGHT, opacity=.78,
        hoverinfo="skip", showlegend=False, legendgroup="real",
    ), **extra)
    fig.add_trace(go.Bar(
        x=centers + widths * .32, y=heights, width=widths * .095,
        marker_color=DIAMONDS_SHADOW, opacity=.54,
        hoverinfo="skip", showlegend=False, legendgroup="real",
    ), **extra)
    fig.add_trace(go.Scatter(
        x=centers, y=heights, mode="markers",
        marker=dict(symbol="line-ew", size=11, color="#E6EFF2",
                    line=dict(color=DIAMONDS_EDGE, width=1.4)),
        hoverinfo="skip", showlegend=False, legendgroup="real",
    ), **extra)
    return centers


def _diamonds_finish(fig, title, subtitle="", height=480):
    """Тот же Plotly-шаблон, что использован для X/Y/Z газовых сенсоров."""
    _diamonds_main_style(fig, title, subtitle, height)
    fig.update_layout(barmode="overlay", hovermode="closest")
    return fig


def diamonds_draw_distribution(fig, data, col, kind, bins=40, smooth=1):
    x = data[col].to_numpy(dtype=float)
    label = DIAMONDS_LABELS[col]
    if kind == "hist":
        counts, edges = np.histogram(x, bins=int(bins))
        _diamonds_add_pipe(fig, edges, counts, counts)
        if len(x) > 2 and np.ptp(x) > 0 and np.std(x) > 0:
            kde = stats.gaussian_kde(x)
            kde.set_bandwidth(kde.factor * float(smooth))
            grid = np.linspace(float(x.min()), float(x.max()), 350)
            # KDE (1 / unit) -> expected frequency in one histogram bin.
            bin_width = float(edges[1] - edges[0])
            fig.add_trace(go.Scatter(
                x=grid, y=kde(grid) * len(x) * bin_width,
                mode="lines", line=dict(color=DIAMONDS_WINE, width=2.5),
                name="KDE · масштаб частот",
                hovertemplate="Значение: %{x:.3g}<br>Ожидаемая частота: %{y:.2f}<extra>KDE</extra>",
            ))
        _diamonds_finish(fig, f"Распределение · {label}",
                         f"n = {len(x):,} · {int(bins)} интервалов · KDE сопоставлена с частотами", 500)
        fig.update_xaxes(title_text=label)
        fig.update_yaxes(title_text="Число наблюдений")
        fig.add_vline(x=float(np.median(x)), line_color=DIAMONDS_WINE,
                      line_dash="dash", line_width=1.6)
    elif kind == "box":
        fig.add_trace(go.Box(
            x=x, orientation="h", boxpoints="outliers",
            fillcolor="rgba(146,172,186,.42)",
            line=dict(color=DIAMONDS_EDGE, width=1.5),
            marker=dict(color=DIAMONDS_TEAL, size=3, opacity=.50),
            name=label,
            hovertemplate="Значение: %{x:.4g}<extra></extra>",
        ))
        _diamonds_finish(fig, f"Boxplot · {label}",
                         f"n = {len(x):,} · отмечены наблюдения за пределами усов", 410)
        fig.update_xaxes(title_text=label)
        fig.update_yaxes(showticklabels=False)
    elif kind == "ecdf":
        vals, counts = np.unique(x, return_counts=True)
        fig.add_trace(go.Scatter(
            x=vals, y=np.cumsum(counts) / len(x), mode="lines",
            line=dict(color=DIAMONDS_EDGE, width=2.6, shape="hv"),
            fill="tozeroy", fillcolor="rgba(146,172,186,.13)",
            name="Эмпирическая CDF",
            hovertemplate="Значение ≤ %{x:.4g}<br>Доля: %{y:.2%}<extra></extra>",
        ))
        _diamonds_finish(fig, f"Эмпирическая CDF · {label}",
                         f"n = {len(x):,} · шаги соответствуют наблюдениям", 440)
        fig.update_xaxes(title_text=label)
        fig.update_yaxes(title_text="Доля наблюдений ≤ x", range=[0, 1.02])
    elif kind == "qq":
        if np.ptp(x) == 0:
            fig.add_annotation(x=.5, y=.5, xref="paper", yref="paper",
                               text="Все значения одинаковы: Q–Q неинформативен", showarrow=False)
        else:
            (theoretical, observed), (slope, intercept, _) = stats.probplot(x, dist="norm", fit=True)
            fig.add_trace(go.Scattergl(
                x=theoretical, y=observed, mode="markers",
                marker=dict(size=4, color=DIAMONDS_METAL, opacity=.5),
                name="Эмпирические квантили",
                hovertemplate="Теоретический квантиль: %{x:.3f}<br>Наблюдение: %{y:.4g}<extra></extra>",
            ))
            line_x = np.array([min(theoretical), max(theoretical)])
            fig.add_trace(go.Scatter(
                x=line_x, y=intercept + slope * line_x, mode="lines",
                line=dict(color=DIAMONDS_WINE, width=2.2, dash="dash"),
                name="Нормальная модель",
            ))
        _diamonds_finish(fig, f"Q–Q · {label}",
                         "Сопоставление с нормальным распределением", 460)
        fig.update_xaxes(title_text="Квантили N(0, 1)")
        fig.update_yaxes(title_text=label, rangemode="normal")
    return fig


def _diamonds_generator_figure(reference, generated, bins, mode):
    fig = make_subplots(
        rows=2, cols=2, vertical_spacing=.16, horizontal_spacing=.12,
        subplot_titles=["Масса: эмпирическая плотность", "Цена: эмпирическая плотность",
                        "Сгенерированные пары", "ECDF цены"],
    )
    for column_idx, name in enumerate(["carat", "price"], start=1):
        real = reference[name].to_numpy(dtype=float)
        fake = generated[name].to_numpy(dtype=float)
        edges = np.histogram_bin_edges(np.r_[real, fake], bins=int(bins))
        real_counts, _ = np.histogram(real, bins=edges)
        fake_counts, _ = np.histogram(fake, bins=edges)
        step = np.diff(edges)
        real_density = real_counts / (len(real) * step)
        fake_density = fake_counts / (len(fake) * step)
        centers = _diamonds_add_pipe(
            fig, edges, real_density, real_counts,
            row=1, col=column_idx, showlegend=(column_idx == 1), label="Исходные данные",
        )
        fig.add_trace(go.Scatter(
            x=centers, y=fake_density, mode="lines",
            line=dict(color=DIAMONDS_WINE, width=2.5, shape="hv"),
            name="Генерация", legendgroup="generated", showlegend=(column_idx == 1),
            hovertemplate="Значение: %{x:.3g}<br>Плотность: %{y:.4g}<extra>Генерация</extra>",
        ), row=1, col=column_idx)
        fig.update_xaxes(title_text=DIAMONDS_LABELS[name], row=1, col=column_idx)
        fig.update_yaxes(title_text="Плотность", row=1, col=column_idx)

    scatter_points = generated.sample(min(6000, len(generated)), random_state=DIAMONDS_SEED)
    fig.add_trace(go.Scattergl(
        x=scatter_points.carat, y=scatter_points.price, mode="markers",
        marker=dict(size=5, color=DIAMONDS_METAL, opacity=.45),
        name="Сгенерированные пары", showlegend=False,
        hovertemplate="Масса: %{x:.3g} карат<br>Цена: %{y:,.0f} USD<extra></extra>",
    ), row=2, col=1)
    fig.update_xaxes(title_text="Масса, карат", row=2, col=1)
    fig.update_yaxes(title_text="Цена, USD", row=2, col=1)

    for df, name, color in [
        (reference, "Исходные данные", DIAMONDS_EDGE),
        (generated, "Генерация", DIAMONDS_WINE),
    ]:
        x, counts = np.unique(df.price.to_numpy(dtype=float), return_counts=True)
        fig.add_trace(go.Scatter(
            x=x, y=np.cumsum(counts) / len(df), mode="lines",
            line=dict(color=color, width=2.4, shape="hv"),
            name=name, showlegend=False,
            hovertemplate="Цена ≤ %{x:,.0f} USD<br>Доля: %{y:.2%}<extra></extra>",
        ), row=2, col=2)
    fig.update_xaxes(title_text="Цена, USD", row=2, col=2)
    fig.update_yaxes(title_text="Доля наблюдений", range=[0, 1.02], row=2, col=2)
    _diamonds_finish(fig, "Реальные данные и генератор",
                     f"{len(reference):,} исходных строк · {len(generated):,} сгенерированных пар · {mode}", 820)
    fig.update_layout(legend=dict(orientation="h", x=0, y=-.14))
    fig.update_annotations(font=dict(color=DIAMONDS_INK, size=12))
    # Для Q–Q / scatter общая ось не обнуляется автоматически.
    return fig


def diamonds_chart_panel(data, col="carat", kind="hist", generation=False):
    style = {"description_width": "155px"}
    layout = widgets.Layout(width="620px", max_width="100%")
    common = dict(style=style, layout=layout)
    limits_m = (float(data.carat.min()), float(data.carat.max()))
    limits_p = (int(data.price.min()), int(data.price.max()))
    mass = widgets.FloatRangeSlider(value=limits_m, min=limits_m[0], max=limits_m[1], step=.01,
                                    readout_format=".2f", description="Масса, от–до (карат)",
                                    continuous_update=False, **common)
    price = widgets.IntRangeSlider(value=limits_p, min=limits_p[0], max=limits_p[1], step=1,
                                   description="Цена, от–до (USD)", continuous_update=False, **common)
    cut = widgets.Dropdown(options=["Все"] + sorted(data.cut.dropna().unique().tolist()),
                           value="Все", description="Огранка", **common)
    bins = widgets.IntSlider(value=40, min=10, max=100, step=5,
                             description="Столбцы гистограммы", continuous_update=False, **common)
    smooth = widgets.FloatSlider(value=1, min=.3, max=2.5, step=.1,
                                 description="Сглаживание KDE", continuous_update=False, **common)
    points = widgets.IntSlider(value=6000, min=100, max=30000, step=100,
                               description="Максимум точек", continuous_update=False, **common)
    logarithm = widgets.Checkbox(value=False, description="Логарифмические оси")
    size = widgets.IntSlider(value=5000, min=100, max=30000, step=100,
                             description="Сгенерировать пар", continuous_update=False, **common)
    seed = widgets.BoundedIntText(value=DIAMONDS_SEED, min=0, max=2147483647,
                                  description="Seed", **common)
    mode = widgets.Dropdown(options=[("Совместная", "joint"), ("Независимая", "independent")],
                            description="Режим генерации", **common)
    apply = widgets.Button(description="Применить", layout=widgets.Layout(width="150px"),
                           button_style="info")
    reset = widgets.Button(description="Сбросить", layout=widgets.Layout(width="150px"))
    status = widgets.HTML()
    output_panel = widgets.Output()
    controls = [mass, price, cut]
    if generation:
        controls += [mode, size, seed, bins]
    elif kind == "hist":
        controls += [bins, smooth]
    elif kind == "scatter":
        controls += [points, logarithm]
    defaults = [(w, w.value) for w in controls]

    def pending(change):
        status.value = f'<span style="color:{DIAMONDS_MUTED}">Параметры изменены · нажмите «Применить»</span>'
    for w in controls:
        w.observe(pending, names="value")

    def render(_=None):
        apply.disabled = True
        try:
            base = diamonds_selected_data(data, mass.value, price.value, cut.value)
            with output_panel:
                clear_output(wait=True)
                if len(base) < 3:
                    display(Markdown("В выбранной группе менее трёх записей. График не построен."))
                    status.value = "Недостаточно данных"
                    return
                display(pd.DataFrame({
                    "Записей": [len(base)],
                    "Средняя масса": [base.carat.mean()],
                    "Медианная цена, USD": [base.price.median()],
                }).round(3))
                if generation:
                    g = diamonds_generate_pairs(base, size.value, seed.value, mode.value)
                    fig = _diamonds_generator_figure(base, g, bins.value, "Совместная" if mode.value == "joint" else "Независимая")
                    display(fig)
                    display(diamonds_compare_generation(base, g).round(4))
                    display(Markdown(
                        f"Корреляция исходных данных: {base.carat.corr(base.price):.4f}; "
                        f"генерации: {g.carat.corr(g.price):.4f}."
                    ))
                    g.to_csv(DIAMONDS_RESULTS / "last_interactive_generation.csv", index=False)
                elif kind == "scatter":
                    viewed = base.sample(min(points.value, len(base)), random_state=DIAMONDS_SEED)
                    fig = go.Figure()
                    for idx, (name, group) in enumerate(viewed.groupby("cut", observed=True)):
                        fig.add_trace(go.Scattergl(
                            x=group.carat, y=group.price, mode="markers",
                            marker=dict(size=6, opacity=.50,
                                        color=DIAMONDS_PALETTE[idx % len(DIAMONDS_PALETTE)]),
                            name=str(name),
                            hovertemplate="Масса: %{x:.3g} карат<br>Цена: %{y:,.0f} USD<extra>Огранка " + str(name) + "</extra>",
                        ))
                    _diamonds_finish(fig, "Связь массы и цены",
                                     f"Показано {len(viewed):,} из {len(base):,} записей · цвет — огранка", 540)
                    fig.update_xaxes(title_text="Масса, карат")
                    fig.update_yaxes(title_text="Цена, USD")
                    if logarithm.value:
                        fig.update_xaxes(type="log")
                        fig.update_yaxes(type="log")
                    display(fig)
                    display(Markdown(
                        f"На графике {len(viewed):,} точек; корреляция по всей выбранной "
                        f"группе: {base.carat.corr(base.price):.4f}."
                    ))
                else:
                    fig = go.Figure()
                    diamonds_draw_distribution(fig, base, col, kind, bins.value, smooth.value)
                    display(fig)
            status.value = (f'<span style="color:{DIAMONDS_EDGE}">'
                            f'Применено · {len(base):,} записей</span>')
        except Exception as exc:
            with output_panel:
                display(Markdown("Не удалось построить график: " + str(exc)))
            status.value = "Ошибка построения"
        finally:
            apply.disabled = False

    def restore(_=None):
        for w, value in defaults:
            w.value = value
        render()

    apply.on_click(render)
    reset.on_click(restore)
    display(widgets.VBox(controls + [widgets.HBox([apply, reset]), status, output_panel]))
    render()
    return {"mass": mass, "price": price, "cut": cut,
            "apply": apply, "reset": reset, "status": status, "render": render}


## 1. Загрузка и аудит
Одна строка — запись о бриллианте. Основные признаки: `carat` — масса в каратах, `price` — цена в долларах США. `cut`, `color`, `clarity` описывают огранку, цвет и чистоту.

Проверяем пропуски, типы, невалидные значения и совпадающие записи. Полностью совпадающие строки не удаляем автоматически: без уникального идентификатора нельзя доказать, что это ошибочные повторы. Нулевые геометрические размеры показываем, но не используем как причину удаления из анализа массы и цены.


### Обоснование подхода

Масса и цена выбраны как две количественные характеристики, для которых можно изучить как отдельные распределения, так и распределение пар. Огранка используется для группировок, но сравнение её групп без контроля массы не выделяет причинного эффекта.

[Содержание](#scrollTo=dm-contents) · [→](#scrollTo=dm-section-02)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
# Основной файл из оригинальной дополнительной работы и проверенное зеркало ggplot2.
DIAMONDS_DATA_URL = "https://waf.cs.illinois.edu/discovery/diamonds.csv"
DIAMONDS_FALLBACK_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/ggplot2/diamonds.csv"
diamonds_local = (ROOT / "data" / "raw" / "diamonds.csv") if "ROOT" in globals() else Path("diamonds.csv")
diamonds_local.parent.mkdir(parents=True, exist_ok=True)
if diamonds_local.exists():
    diamonds_raw = pd.read_csv(diamonds_local)
else:
    try:
        diamonds_raw = pd.read_csv(DIAMONDS_DATA_URL)
        diamonds_download_source = DIAMONDS_DATA_URL
    except Exception as original_error:
        print("Основной CSV недоступен; пробуем зеркало того же набора ggplot2:", original_error)
        diamonds_raw = pd.read_csv(DIAMONDS_FALLBACK_URL)
        diamonds_download_source = DIAMONDS_FALLBACK_URL
    diamonds_raw.to_csv(diamonds_local, index=False)
    print("Данные Diamonds получены по ссылке:", diamonds_download_source)
diamonds_raw = diamonds_raw.drop(columns=[c for c in diamonds_raw if c.startswith("Unnamed:") or c=="rownames"], errors="ignore")
assert {"carat", "price", "cut", "color", "clarity"}.issubset(diamonds_raw.columns), "CSV не содержит нужных колонок"
assert len(diamonds_raw) == 53940, f"Ожидалось 53940 строк Diamonds, получено {len(diamonds_raw)}"
print("Размер исходной таблицы:", diamonds_raw.shape)
print("SHA256 локального CSV:", hashlib.sha256(diamonds_local.read_bytes()).hexdigest())
display(diamonds_raw.head())
display(pd.DataFrame({"тип": diamonds_raw.dtypes.astype(str), "пропуски": diamonds_raw.isna().sum()}))
print("Полностью совпадающие строки сверх первого экземпляра:", diamonds_raw.duplicated().sum())
for col in ["x", "y", "z"]:
    if col in diamonds_raw:
        print(f"Неположительные значения {col}:", (diamonds_raw[col] <= 0).sum())

diamonds_df = diamonds_raw.copy()
for col in ["carat", "price"]:
    diamonds_df[col] = pd.to_numeric(diamonds_df[col], errors="coerce")
valid = np.isfinite(diamonds_df[["carat", "price"]]).all(axis=1) & (diamonds_df["carat"] > 0) & (diamonds_df["price"] > 0)
diamonds_df = diamonds_df.loc[valid].reset_index(drop=True)
DIAMONDS_N = len(diamonds_df)
assert DIAMONDS_N > 2
print(f"Оставлено {DIAMONDS_N:,} строк; исключено {len(diamonds_raw)-DIAMONDS_N:,}.")
display(diamonds_df["cut"].value_counts().rename("число записей").to_frame())
diamonds_df.to_csv(DIAMONDS_RESULTS / "diamonds_analysis.csv", index=False)

In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
display(Markdown(f"""### Обсуждение качества данных
Сохранено {len(diamonds_df):,} записей из {len(diamonds_raw):,}. Найдено {diamonds_raw.duplicated().sum()} совпадающих строк сверх первого экземпляра. Без идентификатора нельзя установить, являются ли они ошибочными дублями, поэтому исследуется распределение записей, а не доказанно уникальных камней.

Нулевые геометрические размеры требуют проверки, но геометрия не входит в выбранные X и Y. Автоматическое удаление по другим столбцам изменило бы целевую совокупность. Контрольная сумма фиксирует версию файла, а не достоверность всех наблюдений.
"""))

## 2. Почему X и Y — случайные величины
Случайный эксперимент — равновероятный выбор одной строки подготовленной таблицы.

$$
\Omega=\{1,\ldots,N\},\quad \mathcal F=2^\Omega,\quad P(\{i\})=1/N.
$$
$$
X(i)=\mathrm{carat}_i,\qquad Y(i)=\mathrm{price}_i.
$$
Для любого борелевского множества $B\subseteq\mathbb R$ прообразы $X^{-1}(B)$ и $Y^{-1}(B)$ являются подмножествами $\Omega$, следовательно, принадлежат $\mathcal F$. Обе функции измеримы и являются случайными величинами.

Случайность связана с выбором строки, а не с тем, что CSV меняется. Ненулевая дисперсия не является определением случайной величины: постоянная функция тоже может быть случайной величиной.

В этой модели распределения дискретны:
$$
P(X=x)=\frac{\#\{i:X(i)=x\}}N.
$$
Далее среднее и дисперсия с `ddof=0` — точные характеристики этой конечной эмпирической модели; дисперсия с `ddof=1` — обычная выборочная статистика.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-01) · [→](#scrollTo=dm-section-03)


### Формулы характеристик
$$
\mu_X=\mathbb E[X]=\frac1N\sum_{i=1}^Nx_i,\quad \operatorname{Var}(X)=\frac1N\sum_{i=1}^N(x_i-\mu_X)^2.
$$
$$
s_X^2=\frac1{n-1}\sum_{i=1}^n(x_i-\bar x)^2,\quad
\widehat F_X(t)=\frac1N\sum_{i=1}^N\mathbf1\{x_i\le t\}.
$$
Знаменатель N описывает дисперсию конечной модели; n−1 используется для стандартной выборочной оценки. Постоянная функция тоже является случайной величиной: ненулевая дисперсия не входит в определение.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-02) · [→](#scrollTo=dm-section-04)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
def diamonds_describe_variable(s):
    q1, q3 = s.quantile([.25,.75])
    return pd.Series({"n": s.size, "среднее": s.mean(), "медиана": s.median(),
        "дисперсия модели (ddof=0)": s.var(ddof=0),
        "выборочная дисперсия (ddof=1)": s.var(ddof=1),
        "выборочное SD": s.std(ddof=1), "min": s.min(), "Q1": q1,
        "Q3": q3, "IQR": q3-q1, "max": s.max(),
        "асимметрия": s.skew(), "эксцесс": s.kurt()})
diamonds_summary = diamonds_df[["carat", "price"]].apply(diamonds_describe_variable)
display(diamonds_summary.round(4))
diamonds_summary.to_csv(DIAMONDS_RESULTS / "descriptive_statistics.csv")
for col, unit in [("carat", "карат"), ("price", "USD")]:
    s = diamonds_df[col]
    display(Markdown(f"**{col}:** среднее {s.mean():.3f} {unit}, медиана {s.median():.3f} {unit}, "
        f"SD {s.std(ddof=1):.3f}. Асимметрия {s.skew():.3f}. "
        + ("Среднее выше медианы; это согласуется с влиянием высоких значений." if s.mean()>s.median() else
           "Среднее не выше медианы; форму распределения уточняем по графикам.")))

In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
display(Markdown(f"""### Интерпретация
Средняя цена {diamonds_df.price.mean():,.2f} USD, медианная {diamonds_df.price.median():,.2f} USD. Высокие значения влияют на среднее, поэтому типичную запись полезно описывать также медианой.

Центральные 50% цен: {diamonds_df.price.quantile(.25):,.2f}–{diamonds_df.price.quantile(.75):,.2f} USD; масс: {diamonds_df.carat.quantile(.25):.2f}–{diamonds_df.carat.quantile(.75):.2f} карата. Это интервалы разброса, не доверительные интервалы среднего. Асимметрия массы {diamonds_df.carat.skew():.3f}, цены {diamonds_df.price.skew():.3f}; конкретный теоретический закон из этого не следует.
"""))

## 3. Одномерный EDA
Гистограмма показывает частоты по интервалам; KDE — сглаженное описание, а не доказанная теоретическая плотность. ECDF не требует выбора интервалов. Boxplot отмечает крайние значения по правилу 1,5 IQR, но не доказывает, что они ошибочны. Q–Q plot сравнивает квантили с нормальной моделью.


### Обоснование подхода

Гистограмма зависит от разбиения на интервалы, поэтому её дополняет ECDF без такого разбиения. KDE описывает сглаженную форму, boxplot — медиану и разброс, а Q–Q — соответствие квантилей нормальной модели. Фильтрация меняет исследуемую совокупность, тогда как число интервалов и сглаживание меняют лишь представление.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-03) · [→](#scrollTo=dm-section-05)


### Гистограмма и KDE: Масса, карат

Гистограмма показывает концентрацию записей по интервалам. KDE — сглаживание, не доказанный закон; пики могут зависеть от округления и состава ассортимента.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-04) · [→](#scrollTo=dm-section-06)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
panel_carat_hist=diamonds_chart_panel(diamonds_df,"carat","hist")


### Boxplot: Масса, карат

Правило 1,5 IQR выделяет крайние значения, а не ошибки. Дорогие или крупные камни не удаляем автоматически.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-05) · [→](#scrollTo=dm-section-07)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
panel_carat_box=diamonds_chart_panel(diamonds_df,"carat","box")


### Эмпирическая функция распределения: Масса, карат

ECDF показывает долю записей ниже порога. В отличие от гистограммы, она не зависит от выбора интервалов.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-06) · [→](#scrollTo=dm-section-08)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
panel_carat_ecdf=diamonds_chart_panel(diamonds_df,"carat","ecdf")


### Q–Q plot: Масса, карат

Отклонения от прямой показывают несовпадение квантилей с нормальной моделью. Это диагностика, не сертификат нормальности.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-07) · [→](#scrollTo=dm-section-09)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
panel_carat_qq=diamonds_chart_panel(diamonds_df,"carat","qq")


### Гистограмма и KDE: Цена, USD

Гистограмма показывает концентрацию записей по интервалам. KDE — сглаживание, не доказанный закон; пики могут зависеть от округления и состава ассортимента.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-08) · [→](#scrollTo=dm-section-10)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
panel_price_hist=diamonds_chart_panel(diamonds_df,"price","hist")


### Boxplot: Цена, USD

Правило 1,5 IQR выделяет крайние значения, а не ошибки. Дорогие или крупные камни не удаляем автоматически.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-09) · [→](#scrollTo=dm-section-11)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
panel_price_box=diamonds_chart_panel(diamonds_df,"price","box")


### Эмпирическая функция распределения: Цена, USD

ECDF показывает долю записей ниже порога. В отличие от гистограммы, она не зависит от выбора интервалов.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-10) · [→](#scrollTo=dm-section-12)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
panel_price_ecdf=diamonds_chart_panel(diamonds_df,"price","ecdf")


### Q–Q plot: Цена, USD

Отклонения от прямой показывают несовпадение квантилей с нормальной моделью. Это диагностика, не сертификат нормальности.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-11) · [→](#scrollTo=dm-section-13)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
panel_price_qq=diamonds_chart_panel(diamonds_df,"price","qq")


### Проверка крайних значений

Границы IQR приведены как диагностика. Все наблюдения сохраняются в основной модели.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-12) · [→](#scrollTo=dm-section-14)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
diamonds_outliers = []
for col in ["carat", "price"]:
    s = diamonds_df[col]
    q1,q3 = s.quantile([.25,.75]); iqr=q3-q1
    flag = (s<q1-1.5*iqr)|(s>q3+1.5*iqr)
    diamonds_outliers.append({"признак":col,"нижняя граница":q1-1.5*iqr,"верхняя граница":q3+1.5*iqr,
                     "число за границами":int(flag.sum()),"доля":flag.mean()})
display(pd.DataFrame(diamonds_outliers))
display(Markdown("Крайние значения сохранены. Автоматическое удаление дорогих или крупных камней изменило бы исследуемое распределение. Отклонения Q–Q от прямой оцениваем визуально; нормальность не предполагаем по умолчанию."))

## 4. Совместный EDA и группировки

Цвет обозначает огранку. Диапазоны массы и цены задают анализируемую группу, а число точек — только объём её отображаемой подвыборки. Корреляция в панели рассчитывается по всей отобранной группе. Логарифмические оси меняют масштаб, но не сами значения.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-13) · [→](#scrollTo=dm-section-15)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
diamonds_panel_joint=diamonds_chart_panel(diamonds_df,kind="scatter")

group_stats = diamonds_df.groupby("cut", observed=True).agg(n=("price","size"),
    mean_carat=("carat","mean"),median_price=("price","median"),mean_price=("price","mean"))
display(group_stats.round(3))
fig_groups = make_subplots(
    rows=1, cols=2, horizontal_spacing=.13,
    subplot_titles=("Цена по огранке", "Масса по огранке"),
)
for pos, col in enumerate(["price", "carat"], start=1):
    fig_groups.add_trace(go.Box(
        x=diamonds_df["cut"], y=diamonds_df[col], boxpoints=False,
        fillcolor="rgba(146,172,186,.45)",
        line=dict(color=DIAMONDS_EDGE, width=1.4),
        marker=dict(color=DIAMONDS_TEAL),
        name=DIAMONDS_LABELS[col], showlegend=False,
        hovertemplate="Огранка: %{x}<br>Значение: %{y:,.3g}<extra></extra>",
    ), row=1, col=pos)
    fig_groups.update_xaxes(title_text="Огранка", row=1, col=pos)
    fig_groups.update_yaxes(title_text=DIAMONDS_LABELS[col], row=1, col=pos)
_diamonds_finish(fig_groups, "Характеристики по огранке",
                 "Крайние точки скрыты только на графике; статистики рассчитаны по всем строкам", 520)
fig_groups.update_annotations(font=dict(color=DIAMONDS_INK, size=12))
display(fig_groups)
display(Markdown("Различие средних цен между огранками не измеряет чистый эффект огранки: группы могут различаться по массе, цвету и чистоте. Эти данные наблюдательные, причинный вывод не делаем."))

### Обсуждение совместных графиков
Средняя цена по огранке отражает состав группы. Группы могут различаться по массе, цвету и чистоте, поэтому разницу средних нельзя приписать только огранке. Разброс цены при близкой массе показывает, что одной массы недостаточно для полного описания.

Логарифмические оси помогают увидеть относительные различия. Более прямолинейная форма не доказывает степенного закона, нормальности или причинности.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-14) · [→](#scrollTo=dm-section-16)


## 5. Точное доказательство зависимости в эмпирической модели
Независимость требует $P(A\cap B)=P(A)P(B)$ для любых событий вида $A=\{X\in C\}$, $B=\{Y\in D\}$.

Берём заранее определённые правила порогов: $a=\operatorname{median}(X)$, $b=\operatorname{median}(Y)$, $A=\{X>a\}$, $B=\{Y>b\}$.

Если $n_A,n_B,n_{AB}$ — числа соответствующих строк, независимость требует
$$
Nn_{AB}=n_An_B.
$$
Проверяем равенство целых чисел, без ошибок округления. Один контрпример достаточен для доказательства зависимости. Совпадение для одной пары событий не доказывает независимость.


### Обоснование подхода

Медианные пороги выбраны как воспроизводимое правило, не требующее произвольной цены или массы. Для опровержения независимости достаточно одного нарушенного равенства вероятностей. Обратный вывод по одному совпадению неверен.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-15) · [→](#scrollTo=dm-section-17)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
DIAMONDS_N = int(len(diamonds_df))
a,b = diamonds_df["carat"].median(), diamonds_df["price"].median()
A,B = diamonds_df["carat"]>a, diamonds_df["price"]>b
nA,nB,nAB = int(A.sum()),int(B.sum()),int((A&B).sum())
left,right = DIAMONDS_N*nAB,nA*nB
diamonds_probabilities = pd.Series({"a, карат":a,"b, USD":b,"P(A)":nA/DIAMONDS_N,"P(B)":nB/DIAMONDS_N,
    "P(A ∩ B)":nAB/DIAMONDS_N,"P(A)P(B)":nA*nB/DIAMONDS_N**2,"разность":(left-right)/DIAMONDS_N**2})
display(diamonds_probabilities.to_frame("значение"))
print("N*nAB =",left,"; nA*nB =",right)
if left != right:
    diamonds_dependence_text = (f"Получено {left} ≠ {right}. Следовательно, X и Y зависимы в модели "
                       "равновероятного выбора строки подготовленного датасета. Это точный вывод для этой модели, без p-value.")
else:
    diamonds_dependence_text = "Для выбранных событий равенство выполнено. Этот расчёт не установил ни зависимость, ни независимость."
display(Markdown(diamonds_dependence_text))
diamonds_probabilities.to_csv(DIAMONDS_RESULTS/"event_probabilities.csv")

In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
display(Markdown(f"""### Смысл доказательства
Вероятность цены выше {b:,.0f} USD равна {nB/DIAMONDS_N:.4f}; среди записей с массой выше {a:.2f} карата — {nAB/nA if nA else float('nan'):.4f}. Знание события о массе меняет вероятность события о цене.

{diamonds_dependence_text}

Вероятности конечной модели известны по полной таблице, поэтому здесь не нужен p-value. Перенос на весь рынок потребовал бы обоснования сбора данных. Равенство для одной пары событий не доказывает независимость.
"""))

## 6. Корреляции: дополнительная характеристика связи
Пирсон измеряет линейную, Спирмен — ранговую монотонную связь. Они не заменяют определение независимости. Проверка нулевой корреляции не является универсальным тестом независимости.

Ниже основной результат — размеры коэффициентов. P-value Пирсона выводится справочно: стандартная модель теста предполагает независимые наблюдения и дополнительные условия. Для этой исторической таблицы случайный отбор с рынка не установлен, поэтому не используем p-value как доказательство для всего рынка.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-16) · [→](#scrollTo=dm-section-18)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
r, p_pearson = stats.pearsonr(diamonds_df["carat"],diamonds_df["price"])
rho, _ = stats.spearmanr(diamonds_df["carat"],diamonds_df["price"])
print(f"Пирсон r = {r:.6f}")
print("Справочное p-value ниже численной точности, а не строго равно нулю." if p_pearson == 0 else f"Справочное p-value: {p_pearson:.3e}")
print(f"Спирмен rho = {rho:.6f}")
rows=[]
for name,g in diamonds_df.groupby("cut", observed=True):
    rows.append({"cut":name,"n":len(g),"Pearson":g["carat"].corr(g["price"]),
                 "Spearman":g["carat"].corr(g["price"],method="spearman")})
diamonds_corr_groups=pd.DataFrame(rows)
display(diamonds_corr_groups.round(4))
diamonds_corr_groups.to_csv(DIAMONDS_RESULTS/"correlations_by_cut.csv",index=False)
display(Markdown(f"Для всей таблицы r={r:.3f}, а ранговая корреляция равна {rho:.3f}. "
    "Различие показателей нужно сопоставить с формой облака точек. Корреляция не устанавливает причинность."))

In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
display(Markdown(f"""### Интерпретация корреляций
Пирсон {r:.4f}, Спирмен {rho:.4f}. Ранговая связь выражена сильнее линейной по этим показателям; форму зависимости уточняем по диаграмме, не объявляем конкретную функцию автоматически.

Внутри огранок Пирсон находится в диапазоне {diamonds_corr_groups.Pearson.min():.4f}–{diamonds_corr_groups.Pearson.max():.4f}. Связь не сводится исключительно к смешению этих групп, но цвет и чистота не контролируются. Корреляция не доказывает причинность; малое p-value не измеряет практическую силу связи.
"""))

## 7. Генератор и его математическое обоснование
**Совместный режим:** независимые индексы $I_k\sim\mathrm{Uniform}\{1,\ldots,N\}$; возвращаем $(X(I_k),Y(I_k))$. Это выборка с возвращением из эмпирического совместного распределения.

**Независимый режим:** независимые индексы $I_k,J_k$; возвращаем $(X(I_k),Y(J_k))$. Тогда
$$
P(X^*\in C,Y^*\in D)=P(X^*\in C)P(Y^*\in D).
$$
Распределения отдельных координат совпадают с исходными эмпирическими распределениями, но совместное распределение заменяется произведением маргинальных.

Это генератор псевдослучайных данных на основе датасета, не криптографический генератор. Одинаковые параметры, данные и seed дают одинаковый результат. В конкретной конечной реализации независимого режима выборочная корреляция обычно не равна строго нулю.


### Обоснование подхода

Эмпирическое ресэмплирование выбрано, чтобы не навязывать данным неподтверждённый параметрический закон. Оно воспроизводит известное распределение, но не создаёт значения вне исходной опоры и не экстраполирует рынок.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-17) · [→](#scrollTo=dm-section-19)


### Законы генераторов
$$
\widehat P_{XY}=\frac1N\sum_i\delta_{(x_i,y_i)},\qquad P_{\mathrm{ind}}=\widehat P_X\otimes\widehat P_Y.
$$
Совместный режим выбирает исходные пары, независимый — координаты раздельно. Маргинальные законы совпадают, но конечные реализации колеблются. Независимый режим может создавать отсутствовавшие в файле пары и не является обоснованной моделью рыночных предложений.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-18) · [→](#scrollTo=dm-section-20)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
def diamonds_generate_pairs(data, n=5000, seed=42, mode="joint"):
    if len(data)==0 or n<1:
        raise ValueError("Нужны непустые данные и n >= 1")
    if mode not in {"joint", "independent"}:
        raise ValueError("Неизвестный режим")
    rng=np.random.default_rng(seed)
    values=data[["carat","price"]].to_numpy()
    i=rng.integers(0,len(values),size=n)
    j=i if mode=="joint" else rng.integers(0,len(values),size=n)
    return pd.DataFrame({"carat":values[i,0],"price":values[j,1]})

def diamonds_compare_generation(reference, generated):
    rows=[]
    for col in ["carat","price"]:
        x,y=reference[col],generated[col]
        rows.append({"признак":col,"среднее исходное":x.mean(),"среднее генерации":y.mean(),
            "SD исходное":x.std(ddof=1),"SD генерации":y.std(ddof=1),
            "медиана исходная":x.median(),"медиана генерации":y.median(),
            "расстояние Вассерштейна":stats.wasserstein_distance(x,y)})
    return pd.DataFrame(rows)

diamonds_joint = diamonds_generate_pairs(diamonds_df,10000,DIAMONDS_SEED,"joint")
diamonds_independent = diamonds_generate_pairs(diamonds_df,10000,DIAMONDS_SEED,"independent")
for name,g in [("Совместная",diamonds_joint),("Независимая",diamonds_independent)]:
    print(name,"генерация; r =",round(g["carat"].corr(g["price"]),4))
    display(diamonds_compare_generation(diamonds_df,g).round(4))
    g.to_csv(DIAMONDS_RESULTS/("generated_joint.csv" if name=="Совместная" else "generated_independent.csv"),index=False)
fig_compare = make_subplots(
    rows=1, cols=3, horizontal_spacing=.055,
    subplot_titles=("Исходные данные", "Совместная генерация", "Независимая генерация"),
)
for pos, (name, group) in enumerate([
    ("Исходные данные", diamonds_df),
    ("Совместная генерация", diamonds_joint),
    ("Независимая генерация", diamonds_independent),
], start=1):
    view = group.sample(min(4000, len(group)), random_state=DIAMONDS_SEED)
    fig_compare.add_trace(go.Scattergl(
        x=view["carat"], y=view["price"], mode="markers",
        marker=dict(size=5, color=DIAMONDS_PALETTE[(pos - 1) % len(DIAMONDS_PALETTE)], opacity=.42),
        name=name, showlegend=False,
        hovertemplate="Масса: %{x:.3g} карат<br>Цена: %{y:,.0f} USD<extra></extra>",
    ), row=1, col=pos)
    fig_compare.update_xaxes(title_text="Масса, карат", row=1, col=pos)
    fig_compare.update_yaxes(title_text="Цена, USD", row=1, col=pos)
_diamonds_finish(fig_compare, "Сравнение двух алгоритмов генерации",
                 "По 4 000 точек на панели · генераторы отличаются сохранением совместного распределения", 560)
fig_compare.update_annotations(font=dict(color=DIAMONDS_INK, size=12))
display(fig_compare)

# Сохраняем иллюстрацию в PNG для воспроизводимого экспорта, без второго вывода в Notebook.
from src.plot_style import apply_plot_style, BLUE, TEAL, BURGUNDY
apply_plot_style()
_png_figure, _png_axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
for ax, (name, group), color in zip(_png_axes, [
    ("Исходные данные", diamonds_df),
    ("Совместная генерация", diamonds_joint),
    ("Независимая генерация", diamonds_independent),
], [BLUE, TEAL, BURGUNDY]):
    view = group.sample(min(4000, len(group)), random_state=DIAMONDS_SEED)
    ax.scatter(view["carat"], view["price"], s=7, alpha=.3, c=color)
    ax.set(title=name, xlabel="Масса, карат", ylabel="Цена, USD")
_png_figure.tight_layout()
_png_figure.savefig(DIAMONDS_RESULTS / "generation_comparison.png", dpi=160)
plt.close(_png_figure)


### Почему средних недостаточно
Разные распределения могут иметь одинаковые средние. Сравниваем также ECDF, квантили, гистограммы и расстояние Вассерштейна. Последнее имеет единицы признака: USD и караты напрямую несопоставимы.

Воспроизведение отдельных столбцов не гарантирует воспроизведения пар. Ресэмплирование не создаёт новых значений за пределами опоры исходного файла и не экстраполирует рынок.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-19) · [→](#scrollTo=dm-section-21)


## 8. Интерактивный генератор

Диапазоны массы, цены и огранка задают исходную группу. Размер определяет число сгенерированных пар, seed — воспроизводимую случайную реализацию, а режим — сохранение или разрушение зависимости координат. Число столбцов меняет только разбиение гистограмм.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-20) · [→](#scrollTo=dm-section-22)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
panel_generator=diamonds_chart_panel(diamonds_df,generation=True)


## 9. Контроль генератора и сходимость оценок
Автотесты проверяют форму, воспроизводимость, принадлежность значений исходной опоре и сохранение пар совместным генератором. Мы не требуем нулевой корреляции от каждой независимой конечной выборки.

В симуляции оцениваем RMSE среднего по повторным выборкам с возвращением. При таком отборе теоретическая стандартная ошибка среднего равна SD эмпирической модели / √n. Это исследование генератора из конечной таблицы, а не доказательство репрезентативности таблицы для рынка.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-21) · [→](#scrollTo=dm-section-23)


### Теория ошибки среднего
$$
\mathbb E[\bar Y_n]=\mu_Y,\quad\operatorname{Var}(\bar Y_n)=\sigma_Y^2/n,\quad\operatorname{SE}(\bar Y_n)=\sigma_Y/\sqrt n.
$$
$$
\operatorname{RMSE}(n)=\sqrt{\frac1R\sum_{r=1}^R(\bar Y_n^{(r)}-\mu_Y)^2}.
$$
Чтобы уменьшить теоретическую SE вдвое, нужно увеличить n вчетверо. Симуляционная RMSE при конечном числе повторов колеблется и не обязана точно совпадать с теорией. Большая генерация снижает шум относительно таблицы, но не устраняет смещения её сбора.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-22) · [→](#scrollTo=dm-section-24)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
test_a=diamonds_generate_pairs(diamonds_df,100,42,"joint")
assert test_a.equals(diamonds_generate_pairs(diamonds_df,100,42,"joint"))
assert test_a.shape==(100,2)
assert test_a["carat"].isin(diamonds_df["carat"]).all()
assert test_a["price"].isin(diamonds_df["price"]).all()
support=set(map(tuple,diamonds_df[["carat","price"]].to_numpy()))
assert all(tuple(row) in support for row in test_a.to_numpy())
assert not test_a.equals(diamonds_generate_pairs(diamonds_df,100,43,"joint"))
print("Все базовые тесты пройдены.")

rng=np.random.default_rng(DIAMONDS_SEED)
repetitions=200
diamonds_simulation=[]
values=diamonds_df["price"].to_numpy()
for n in [50,100,300,1000,3000,10000]:
    means=np.array([rng.choice(values,size=n,replace=True).mean() for _ in range(repetitions)])
    diamonds_simulation.append({"n":n,"среднее оценок":means.mean(),
        "RMSE среднего":np.sqrt(np.mean((means-values.mean())**2)),
        "теоретическая SE":values.std(ddof=0)/np.sqrt(n)})
diamonds_simulation=pd.DataFrame(diamonds_simulation)
display(diamonds_simulation.round(3))
fig_convergence = go.Figure()
fig_convergence.add_trace(go.Scatter(
    x=diamonds_simulation["n"], y=diamonds_simulation["RMSE среднего"],
    mode="lines+markers", marker=dict(size=9, color=DIAMONDS_EDGE),
    line=dict(color=DIAMONDS_EDGE, width=2.7), name="RMSE · 200 повторений",
    hovertemplate="n = %{x:,.0f}<br>RMSE: %{y:.3g} USD<extra></extra>",
))
fig_convergence.add_trace(go.Scatter(
    x=diamonds_simulation["n"], y=diamonds_simulation["теоретическая SE"],
    mode="lines+markers", marker=dict(size=8, color=DIAMONDS_WINE, symbol="square"),
    line=dict(color=DIAMONDS_WINE, width=2.2, dash="dash"), name="Теоретическая SD / √n",
    hovertemplate="n = %{x:,.0f}<br>SE: %{y:.3g} USD<extra></extra>",
))
_diamonds_finish(fig_convergence, "Точность средней цены при увеличении n",
                 "200 повторных выборок с возвращением при каждом объёме", 490)
fig_convergence.update_xaxes(title_text="Размер генерации n · логарифмический масштаб", type="log")
fig_convergence.update_yaxes(title_text="Ошибка среднего, USD · логарифмический масштаб",
                             type="log", rangemode="normal")
display(fig_convergence)
diamonds_simulation.to_csv(DIAMONDS_RESULTS / "simulation.csv", index=False)


## 10. Итоговые выводы
Следующая ячейка формирует выводы из полученных результатов. В отчёте различаем точное доказательство зависимости в эмпирической модели, описательные корреляции и ограничения переноса результатов.

[Содержание](#scrollTo=dm-contents) · [←](#scrollTo=dm-section-23)


In [ ]:
# @title Diamonds · расчёты { display-mode: "form" }
diamonds_conclusion = f"""# Итоговые выводы

1. Подготовлено {DIAMONDS_N:,} строк. Удалено {len(diamonds_raw)-DIAMONDS_N:,} записей с некорректной массой или ценой; совпадающие строки автоматически не удалялись.
2. X — масса в каратах, Y — цена в USD. Обе функции являются случайными величинами на конечном вероятностном пространстве равновероятного выбора строки; измеримость доказана в разделе 2.
3. Средняя масса: {diamonds_df.carat.mean():.4f} карата, медианная: {diamonds_df.carat.median():.4f}. Средняя цена: {diamonds_df.price.mean():.2f} USD, медианная: {diamonds_df.price.median():.2f} USD.
4. {diamonds_dependence_text}
5. Корреляция Пирсона: {r:.4f}; Спирмена: {rho:.4f}. Эти показатели описывают связь, но не причинность.
6. Совместный генератор воспроизводит эмпирическое распределение пар; независимый сохраняет маргинальные распределения в законе генерации, но заменяет совместный закон их произведением.
7. В реализации из 10000 пар корреляция совместного генератора составила {diamonds_joint.carat.corr(diamonds_joint.price):.4f}, независимого — {diamonds_independent.carat.corr(diamonds_independent.price):.4f}. Конечные реализации колеблются; это не ошибка генератора.
8. Ограничения: исторические наблюдательные данные, неустановленная репрезентативность для всего рынка, возможные повторные записи, влияние огранки, цвета и чистоты. Нормальность и причинность не предполагаются.
"""
display(Markdown(diamonds_conclusion))
(DIAMONDS_RESULTS/"conclusions.md").write_text(diamonds_conclusion,encoding="utf-8")
display(Markdown(f"""## Обсуждение и ограничения
Разделены определение модели, её точные свойства и описательные закономерности исторических данных. Различие средней цены ({diamonds_df.price.mean():,.2f} USD) и медианы ({diamonds_df.price.median():,.2f} USD) показывает, почему одной средней недостаточно.

Зависимость установлена через вероятности событий. Корреляции Пирсона ({r:.4f}) и Спирмена ({rho:.4f}) дополняют вывод, но не объясняют причинный механизм. Генераторы показывают, что сохранение одномерных распределений не равно сохранению совместного закона.

Ограничения: исторические цены, неизвестная репрезентативность, возможные повторные записи, неоднородность качества. Универсальная формула цены и причинный эффект массы не установлены. Дальнейшие исследования могли бы включать узкие диапазоны массы, стратификацию по цвету и чистоте, чувствительность к повторам и внешний набор; они пока не выполнены.
"""))

diamonds_final_text=f"""## Заключение

Средняя цена равна {diamonds_df.price.mean():,.2f} USD, медианная — {diamonds_df.price.median():,.2f} USD. Центральная половина цен находится между {diamonds_df.price.quantile(.25):,.2f} и {diamonds_df.price.quantile(.75):,.2f} USD. Эти показатели дополняют друг друга: одна средняя не описывает неоднородность записей.

Главный результат сравнения генераторов — сохранение отдельных распределений не означает сохранения сочетаний признаков. Совместный режим воспроизводит исходный эмпирический закон пар, независимый — произведение маргинальных законов. Поэтому похожие гистограммы сами по себе не гарантируют реалистичных сочетаний массы и цены.

Фильтры интерактивных панелей меняют состав группы. Корреляция в узком диапазоне и по всей таблице отвечает на разные вопросы; различие нельзя автоматически считать ошибкой или причинным эффектом. Итоговые показатели этого раздела относятся ко всей подготовленной таблице и не перезаписываются фильтрами.

Исследование даёт воспроизводимую модель исторического файла, а не универсальную формулу цены. Для рыночного прогноза потребовались бы актуальные данные, обоснованный отбор и отдельная проверка точности. Увеличение размера генерации уменьшает случайные колебания относительно файла, но не устраняет ограничения его сбора.
"""
display(Markdown(diamonds_final_text))
(DIAMONDS_RESULTS/"conclusions.md").write_text(diamonds_conclusion+"\n"+diamonds_final_text,encoding="utf-8")


## Источники

- [Описание diamonds и признаков — ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html).
- [Исходный CSV](https://waf.cs.illinois.edu/discovery/diamonds.csv).


### Результат дополнительного исследования

- Рассмотрены две случайные величины на конечной модели равновероятного выбора записи.
- Проведены одномерный и совместный EDA, диагностика крайних значений, количественное исследование зависимости и корреляций.
- Сопоставлены **совместный** и **независимый** генераторы пар `carat`–`price`; выполнены тесты и эксперимент со сходимостью среднего.
- Реализованы интерактивные панели выбора огранки, диапазонов массы и цены, параметров визуализации и генератора.
- Итоговые числовые выводы вычисляются из загруженного Diamonds и не смешиваются с оценками газовых сенсоров.

[Перейти к `cat_mark_table`](#scrollTo=0e7069b9) · [Источник ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html)


## Источники и ограничения исследования

1. **Первичные измерения.** [UCI Machine Learning Repository: Gas Sensor Array Drift at Different Concentrations (ID 270)](https://archive.ics.uci.edu/dataset/270/gas+sensor+array+drift+at+different+concentrations), DOI: [10.24432/C5MK6M](https://doi.org/10.24432/C5MK6M). В описании указан официальный код газа: **5 = Acetone**.
2. **Экспериментальная методика и дрейф.** Vergara A. et al. (2012). *Chemical gas sensor drift compensation using classifier ensembles*. Sensors and Actuators B, 166–167, 320–329. DOI: [10.1016/j.snb.2012.01.074](https://doi.org/10.1016/j.snb.2012.01.074).
3. **Планирование сенсорных экспериментов.** Rodríguez-Luján I. et al. (2014). *On the calibration of sensor arrays for pattern recognition using the minimal number of experiments*. Chemometrics and Intelligent Laboratory Systems, 130, 123–134. DOI: [10.1016/j.chemolab.2013.10.012](https://doi.org/10.1016/j.chemolab.2013.10.012).
4. **Дополнительный датасет.** [Diamonds — официальная документация ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html) (53 940 записей, 10 переменных).
5. **Исходный CSV Diamonds.** [University of Illinois](https://waf.cs.illinois.edu/discovery/diamonds.csv), резервная копия: [Rdatasets / ggplot2](https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/ggplot2/diamonds.csv).

**Ограничения.** Данные исторические и наблюдательные. Строки внутри партии могут быть зависимы; корреляции не устанавливают причинность; генератор моделирует распределение признака, а не химическую кинетику. Исследование Diamonds имеет **источник ggplot2**, а не ResearchGate, и не смешивается с газовыми измерениями.


## Lab resume · сводный вывод

1. Полное исходное исследование X, Y, Z сохранено, включая анализ концентрации, партий, корреляционную матрицу 128 признаков и повторные выборки.
2. При стандартной нормализации меняются масштаб и центр, но не форма эмпирического распределения.
3. Наблюдаемые характеристики могут зависеть от концентрации и экспериментальной партии; визуального сравнения недостаточно для вывода о причине.
4. Корреляции между X, Y, Z подтверждаются вычислениями, но интерпретируются с учётом общих экспериментальных факторов.
5. Сравнение генераторов показывает различие между сохранением эмпирической формы и приближением теоретическими семействами. Нет оснований считать какой-либо генератор физическим описанием датчика без отдельной проверки.
6. Дополнительное исследование IQR-флагов показывает чувствительность выборочных характеристик; потенциальные выбросы не удаляются из основных данных автоматически.
7. Дополнительный датасет Diamonds исследован отдельно: масса и цена как случайные величины, EDA, зависимость, генерация и тесты; его метрики не подменяют характеристики сенсоров.


## cat_mark_table · проверка критериев с переходами

Ведомость с официальными `cat_id` и коэффициентами ещё не предоставлена; ниже отражено фактическое состояние работы.
Ссылки ведут в разделы этого Notebook, а статусы описывают **только проверяемое** выполнение.

### Lab01 — sprint 1

| cat_id / критерий | notebook_link | coeff | comment |
|---|---|---:|---|
| Структура, индексы, якорные ссылки | [Содержание](#scrollTo=8a1564ab) | — | Вложенные заголовки, навигация |
| Три случайные величины | [X](#scrollTo=4edb5f0b), [Y](#scrollTo=b6f3df63), [Z](#scrollTo=5f3e90a9) | — | Авторские исследования сохранены |
| EDA и распределения | [X](#scrollTo=4edb5f0b), [Y](#scrollTo=b6f3df63), [Z](#scrollTo=5f3e90a9) | — | Гистограммы, ECDF, Q–Q, boxplot |
| Масштабы, единицы и дизайн | [Визуализации](#scrollTo=4edb5f0b) | — | Оси подписаны; единицы требуют уточнения по документации |
| Стандартизация | [Математическая основа](#scrollTo=abb921e8) | — | Три z-score |
| Зависимость и гипотеза | [Гипотезы](#scrollTo=c3b732ed) | — | Y–Z, контролируемые факторы |
| Устойчивость оценок | [X](#scrollTo=4edb5f0b), [Y](#scrollTo=b6f3df63), [Z](#scrollTo=5f3e90a9) | — | 300 повторов × 2 схемы для каждой величины |
| Литература | [Источники](#scrollTo=c63e0600) | — | Первоисточники, ResearchGate и UCI |
| LaTeX | [Математика](#scrollTo=abb921e8) | — | Исходные формулы и оценки в Markdown |
| Скрытие кода | [Оформление](#scrollTo=8a1564ab) | — | notebook cell tags/metadata; проверить в Google Colab |

### Lab01 — sprint 2

| cat_id / критерий | notebook_link | coeff | comment |
|---|---|---:|---|
| Генератор на основе данных | [Генерация](#scrollTo=1bcac0f6) | — | Эмпирическая квантильная функция |
| Другие генераторы | [Сравнение](#scrollTo=1bcac0f6) | — | Нормальный и skew-normal квантильные методы |
| Математическое обоснование | [Генерация](#scrollTo=1bcac0f6) | — | Метод обратной функции, ограничения |
| Сравнение метрик | [Сравнение](#scrollTo=1bcac0f6) | — | Среднее, медиана, SD, квантили, KS |
| Автотесты | [Автотесты](#scrollTo=5cd0d137) | — | pytest в среде Colab |
| Интерактивность, marimo | [Интерактивность](#scrollTo=01368f1b) | — | Отдельное приложение; проверить локально |
| Очистка / дополнительный анализ | [Выбросы](#scrollTo=01d1e10d) | — | До/после, без подмены основных данных |
| Дополнительное исследование Diamonds (КТвД) | [Diamonds](#scrollTo=diamonds-intro), [EDA](#scrollTo=dm-section-04), [генераторы](#scrollTo=dm-section-18) | — | Две случайные величины, зависимость, интерактивность и автотесты; источник — официальный ggplot2, не ResearchGate |
### Дополнительная работа КТвД · Diamonds

| cat_id / критерий | notebook_link | coeff | comment |
|---|---|---:|---|
| Две новые случайные величины | [Постановка и обоснование](#scrollTo=dm-section-02) | — | `carat` и `price`, конечная вероятностная модель |
| EDA и интерактивные графики | [Диагностика X/Y](#scrollTo=dm-section-04), [совместный EDA](#scrollTo=dm-section-14) | — | Гистограммы, boxplot, ECDF, Q–Q и фильтры |
| Зависимость | [Доказательство](#scrollTo=dm-section-16), [корреляции](#scrollTo=dm-section-17) | — | Проверка совместных вероятностей событий и коэффициентов |
| Генераторы и тесты | [Генератор](#scrollTo=dm-section-18), [автотесты](#scrollTo=dm-section-22) | — | Совместная/независимая генерация пар и эмпирические проверки |
| Первоисточник данных | [ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html) | — | Официальная документация Diamonds; иной источник, не ResearchGate |


## Проверка критериев КТвД · прямые ссылки

| Критерий | Доказательство в проекте | Проверка / статус |
|---|---|---|
| Структура исследования, заголовки, индексы | [Навигация](#scrollTo=28df4a97), [X](#scrollTo=4edb5f0b), [Y](#scrollTo=b6f3df63), [Z](#scrollTo=5f3e90a9), [Спринт 2](#scrollTo=1bcac0f6) | Реализовано в Notebook |
| Якорные ссылки | [Содержание](#scrollTo=28df4a97), [Визуальный атлас](#scrollTo=3b116392) | Реализовано |
| Скрытие кода | Ячейки Python оформлены с `# @title ... { display-mode: "form" }` | Проверить переключатель *Показать код* в Google Colab |
| Правильные шкалы и единицы | [Распределения](#scrollTo=dcec2215), [Интервалы](#scrollTo=61b8c1e3), [Связи](#scrollTo=2d37d337) | Оси подписаны, все значения из реальных данных |
| Единый стиль | [Дизайн-система](#scrollTo=4e97bb4d), [Визуальный атлас](#scrollTo=3b116392) | Основные диаграммы интерактивны; статические исходные иллюстрации также сохранены |
| Математические формулы LaTeX | [Математическая база](#scrollTo=abb921e8), [Генераторы](#scrollTo=1bcac0f6) | Markdown LaTeX в Jupyter |
| Три случайные величины и EDA | [X](#scrollTo=4edb5f0b), [Y](#scrollTo=b6f3df63), [Z](#scrollTo=5f3e90a9) | Сохранена вся авторская часть |
| Зависимость и гипотеза | [Связи](#scrollTo=2d37d337), [Основной анализ](#scrollTo=dc0e3bfa) | Проверка Y–Z, а не X–Z |
| Эмпирические оценки | [Выборки](#scrollTo=7b4fe614) | Повторные выборки, 300 испытаний |
| Теоретические модели | [Модели](#scrollTo=a16dd2e7), [Дополнительное исследование](#scrollTo=365b68a7) | Кривые и AIC |
| Автотесты | [Автотесты](#scrollTo=5cd0d137) | Проверяются при Run All |
| Генераторы случайных чисел | [Спринт 2](#scrollTo=1bcac0f6) | Расчёты и сравнение метрик |
| Интерактивность в Jupyter | [Интерактивные разделы X/Y/Z](#scrollTo=4edb5f0b), [дополнительная сводная панель](#scrollTo=3b116392) | Реализованы в Plotly |
| marimo | `app/Lab01_KTvD_MARIMO_FINAL.py` | Интерактивное приложение работает отдельно от Google Colab |
| Дополнительное исследование Diamonds (КТвД) | [Открыть Diamonds](#scrollTo=diamonds-intro) · [обоснование](#scrollTo=dm-section-02) · [EDA](#scrollTo=dm-section-04) · [генератор](#scrollTo=dm-section-18) · [источник ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html) | Реализовано в Notebook; источник — ggplot2, не ResearchGate |
| Ссылки на первоисточники | [Источники](#scrollTo=c63e0600) | Указаны; проверить доступность у преподавателя |

**Официальные `cat_id` и коэффициенты `coeff`** отсутствуют в материалах. Ниже — подготовленная форма без вымышленных значений; после получения ведомости замените обозначения и добавьте прямые ссылки на опубликованный GitHub Notebook.

| cat_id | notebook_link | coeff | comment |
|---|---|---|---|
| код преподавателя — структура | [Оглавление](#scrollTo=28df4a97) | — | Индексы, навигация |
| код преподавателя — графики | [Интерактивный атлас](#scrollTo=3b116392) | — | интерактивные графики и сводные панели |
| код преподавателя — LaTeX | [Теория](#scrollTo=abb921e8) | — | Формулы Markdown |
| код преподавателя — генератор | [Спринт 2](#scrollTo=1bcac0f6) | — | Генераторы и проверки |
| код преподавателя — тесты | [Автотесты](#scrollTo=5cd0d137) | — | pytest |
| дополнительное исследование КТвД | [Diamonds](#scrollTo=diamonds-intro) | — | Масса/цена, EDA, зависимость, генерация; официальный источник ggplot2 |

### Финальная проверка дизайнерской части

- [X: интерактивная диагностика](#scrollTo=4edb5f0b) · [Y](#scrollTo=b6f3df63) · [Z](#scrollTo=5f3e90a9) — встроены на местах исходных рисунков.
- [Зависимость и корреляции](#scrollTo=4edb5f0b) — диаграммы рассеяния, интерактивная корреляционная матрица 128 признаков.
- [Сводные интерактивные панели](#scrollTo=3b116392) — дополнительные средства фильтрации и смены параметров.
- `site/LAB01_FINAL_KTvD_INTERACTIVE.html` — отдельная обзорная страница; `app/Lab01_KTvD_MARIMO_FINAL.py` — реактивное приложение.

**Отдельно от ноутбука:** преподавательские коды и коэффициенты `cat_id`/`coeff` не предоставлены; для Diamonds используется открытый источник ggplot2 (не ResearchGate).

**Включённая доп. работа:** [Diamonds — ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html) · [прямой переход к исследованию](#scrollTo=diamonds-intro). Дополнительные расчёты отделены от сенсорного исследования и используют собственные переменные.
